In [1]:
%%writefile HellaSwag.py
import os
import json
import requests
import tiktoken
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import GPT2LMHeadModel


DATA_CACHE_DIR = os.path.join(os.path.dirname(__file__), "HellaSwag")


def download_file(url, fname, chunk_size=1024):
    resp  =  requests.get(url, stream=True)
    total = int(resp.headers.get('content-length', 0)) #  resp.headers -> dictionary of HTTP response headers sent by the server / .get("content-length", 0) -> gets the file size in bytes from the header, returns 0 if header is missing (some servers don't send it)
    with open(fname, 'wb') as f, tqdm(desc=fname, total=total, unit='iB', unit_scale=True, unit_divisor=1024,) as bar:
        for data in resp.iter_content(chunk_size=chunk_size):  #  resp.iter_content(chunk_size=1024) -> yields raw bytes in pieces of 1024 bytes at a time
            size = f.write(data)  # > writes those raw bytes to disk & returns the number of bytes actually written
            bar.update(size) # update how many bytes were just written


hellaswags = {
    "train": "https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_train.jsonl",
    "val": "https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_val.jsonl",
    "test": "https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_test.jsonl",
}


enc = tiktoken.get_encoding("gpt2")


def download(split):
    os.makedirs(DATA_CACHE_DIR, exist_ok=True)
    data_url = hellaswags[split]
    data_filename = os.path.join(DATA_CACHE_DIR, f"hellaswag_{split}.jsonl")
    if not os.path.exists(data_filename):
        print(f"Downloading {split} data from {data_url} to {data_filename}")
        download_file(data_url, data_filename)


def render_example(example):
    # example is a dict from the jsonl file, looks like:
    # {
    #   "ctx": "The woman picked up the ball and",
    #   "label": 2,           <- index of the correct ending (0,1,2, or 3)
    #   "endings": [          <- always exactly 4 possible completions
    #       "threw it away",
    #       "sat down quietly",
    #       "threw it to her friend",
    #       "read a book"
    #   ]
    # }

    ctx = example["ctx"]
    label = example["label"]
    endings = example["endings"]

    data = {
        'label': label,
        'ctx_tokens' : None,
        'endings_tokens' : [],
    }

    ctx_tokens = enc.encode(ctx)
    data["ctx_tokens"] = ctx_tokens

    tok_rows = []
    mask_rows = []
    # tok_rows  -> will hold 4 lists, each being [ctx_tokens + one_ending_tokens]
    # mask_rows -> will hold 4 lists, each being [0,0,0,...,1,1,1]
    #              zeros over the context, ones over the ending,  so we can later evaluate loss ONLY on the ending part



    for ending in endings:
        end_tokens = enc.encode(' ' + ending)
        # prepends a space before the ending before tokenizing
        # this is because GPT-2's tokenizer was trained on text where
        # words after a space get different token ids than words at the start
        # e.g. "friend" and " friend" are different tokens in GPT-2's vocab
        # so prepending the space gives the correct tokenization for a word
        # that would naturally follow the context

        tok_rows.append(ctx_tokens + end_tokens)
        mask_rows.append(len(ctx_tokens)*[0] + len(end_tokens)*[1])

        data['endings_tokens'].append(end_tokens)       # store each ending's tokens for debugging


    max_len = max(len(row) for row in tok_rows)     # needed because the 4 endings have different numbers of tokens, and we need all rows to be the same length to form a tensor

    tokens = torch.zeros((4, max_len), dtype=torch.long) # we have 4 choices in each question
    mask = torch.zeros((4, max_len), dtype=torch.long)

    for i, (tok_row, mask_row) in enumerate(zip(tok_rows, mask_rows)):
        tokens[i, :len(tok_row)] = torch.tensor(tok_row) # tokens[i, :len(tok_row)] -> selects row i, columns 0 to len(tok_row) |   = torch.tensor(tok_row) -> fills those positions with this row's token ids
        mask[i, :len(tok_row)] = torch.tensor(mask_row)      # columns beyond len(tok_row) stay as 0 (padding from torch.zeros above)


    return data, tokens, mask, label
    # data   -> debug dict
    # tokens -> (4, max_len) tensor of token ids, one row per candidate ending
    # mask   -> (4, max_len) tensor of 0s and 1s marking ending positions
    # label  -> integer 0-3, which row is the correct ending


def iterate_examples(split):
    download(split)
    with open(os.path.join(DATA_CACHE_DIR, f"hellaswag_{split}.jsonl"), "r") as f:
        for line in f:
            # iterates over the file one line at a time
            # each `line` is a string like '{"ctx": "...", "label": 2, "endings": [...]}\n'
            example = json.loads(line)
            yield example
            # yield makes this a GENERATOR function instead of a regular function
            # instead of returning all examples at once (which would load everything into RAM),
            # it pauses here and gives back one example at a time
            # the caller gets one example, processes it, then this resumes for the next one
            # memory efficient for large datasets


@torch.no_grad()
def evaluate(model_type, device):
    torch.set_float32_matmul_precision('high')     # tells PyTorch to use TF32 precision on Ampere GPUs (A100, RTX 3090, etc.) | TF32 is faster than full FP32 with negligible accuracy loss | 'high' enables TF32, 'highest' forces full FP32
    model = GPT2LMHeadModel.from_pretrained(model_type)
    model.to(device)
    # loads HuggingFace's pretrained GPT-2 and moves to GPU/CPU
    num_correct_norm = 0   # counts correct predictions using length-normalized loss
    num_correct = 0        # counts correct predictions using raw total loss
    num_total = 0          # counts total examples seen


    for example in iterate_examples("val"):
        # calls our generator above, gets one example dict at a time
        data, tokens, mask, label = render_example(example)
        tokens = tokens.to(device)
        mask = mask.to(device)
        logits = model(tokens).logits # shape: (4, max_len, vocab_size)

        shift_logits = (logits[:,:-1,:]).contiguous()    # .contiguous() -> ensures the tensor's memory layout is sequential after slicing some PyTorch operations require this
                                                         #  removing the last position means we drop the prediction AFTER the sequence ends
        shift_tokens = (tokens[..., 1:]).contiguous()    #  tokens[..., 1:] -> removes the FIRST token, keeps everything from position 1 onward ->  aligns with shift_logits so that position i's logits are compared against token i+1

        shift_logits = shift_logits.view(-1, shift_logits.size(-1))

        shift_tokens = shift_tokens.view(-1)

        loss = F.cross_entropy(shift_logits, shift_tokens, reduction='none')     # reduction='none' -> returns one loss value per position instead of averaging

        reshaped_loss = loss.view(tokens.size(0), -1)

        shift_mask = mask[...,1:].contiguous()

        masked_shift_loss = reshaped_loss * shift_mask

        sum_loss = torch.sum(masked_shift_loss, dim=-1)

        avg_loss = sum_loss / shift_mask.sum(-1)   # shift_mask.sum(dim=1) -> counts how many 1s are in each row,  i.e. how many ending tokens each candidate has, without this, longer endings would always have higher total loss

        preds = torch.argmin(sum_loss).item()

        norm_preds = torch.argmin(avg_loss).item()

        num_total += 1

        num_correct += int(preds == label)

        num_correct_norm += int(norm_preds == label)

        print(f"{num_total} acc_norm: {num_correct_norm}/{num_total}={num_correct_norm / num_total:.4f}")

        if num_total < 10:
            # only prints detailed debug info for the first 9 examples
            print(f"Context:\n {example['ctx']}")
            print(f"Endings:")
            # \n inside the string is a newline character
            for i, end in enumerate(example["endings"]):
                print(f"{i} (loss: {avg_loss[i].item():.4f}) {end}")
                # avg_loss[i] -> indexes into the 4-element tensor to get loss for ending i
                # .item() -> converts tensor element to plain Python float
            print(f"predicted: {norm_preds}, actual: {label}")



if __name__ == "__main__":
# this block ONLY runs if you execute this file directly: python hellaswag.py
# it does NOT run if another file imports this file as a module
# standard Python pattern to separate "runnable script" from "importable module"

    import argparse
    parser = argparse.ArgumentParser()
    # argparse lets you pass command line arguments when running the script
    # e.g. python hellaswag.py -m gpt2-medium -d cuda

    parser.add_argument("-m", "--model_type", type=str, default="gpt2", help="the model type to use")
    # -m is the short flag, --model_type is the long flag, both do the same thing
    # type=str -> convert the input to string
    # default="gpt2" -> if you don't pass -m, it uses "gpt2"
    # help="..." -> shown when you run python hellaswag.py --help

    parser.add_argument("-d", "--device", type=str, default="cuda", help="the device to use")

    args = parser.parse_args()
    # actually reads sys.argv (the command line you typed) and populates args
    # args.model_type -> whatever you passed with -m
    # args.device -> whatever you passed with -d

    evaluate(args.model_type, args.device)
    # calls the main function with the parsed arguments

Writing HellaSwag.py


In [2]:
%%writefile train_gpt2.py
import os
import math
import time
import inspect
import tiktoken
import numpy as np

from torch.distributed import init_process_group, destroy_process_group
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist

from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
from functools import partial
from HellaSwag import render_example, iterate_examples

###_________________________ DISTRIBUTED TRAINING _______________________________________

ddp = int(os.environ.get('RANK', -1)) != -1
if ddp:
    assert torch.cuda.is_available(), f'There is not cude for the device to run ddp'
    init_process_group(backend='nccl')
    ddp_rank = int(os.environ['RANK'])
    ddp_local_rank = int(os.environ['LOCAL_RANK'])
    ddp_world_size = int(os.environ['WORLD_SIZE'])
    device = f'cuda:{ddp_local_rank}'
    torch.cuda.set_device(device)
    master_process = ddp_rank == 0
else:
    ddp_rank = 0
    ddp_local_rank = 0
    ddp_world_size = 1
    master_process = True
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Using device: {device}')


###_________________________ HYPERPARAMETERS _______________________________________

@dataclass
class GPT_config:
    ### for model
    n_embd: int = 1024
    n_head: int = 8
    n_layer: int = 8
    vocab_size: int = 50304
    block_size: int = 512  # Just a number that is divisible by 2 more times that the standard 50257
    ### for data
    bs: int = 16                          # per-GPU micro batch
    tokenizer: str = 'gpt2'
    tot_bs_for_grad_accum: int = 131072       
    # for the training shards
    data_root: str = "/kaggle/input/datasets/seif222/gpt-train-kaggle-zero-to-hero"   # <- point at your actual shards
    ### for optim sched
    max_lr: float = 6e-4
    min_lr_ratio: float = 0.1
    warmup_steps: int = 10
    max_steps: int = 900
    ### for optim
    beta1: float = 0.9
    beta2: float = 0.95
    eps: float = 1e-8
    weight_decay: float = 0.1
    ### for training & Validation
    training_steps: int = 1000                 # = max_steps
    val_after_step: int = 100
    val_loss_accum_steps: int = 5
    ### CheckPoint
    checkpoint_after_steps: int = 50
    ### device
    device: str = device
    ### Multi-Device Training
    process_rank: int = ddp_rank
    num_processes: int = ddp_world_size
    ### Options
    # validaiton
    validation: bool = True
    # sampling
    model_sampling: bool = True
    num_sequence: int = 3
    max_length: int = 40
    #compile
    use_compile: bool = False                # note: compile breaks HellaSwag/gen

###_________________________ CREATING THE MODEL _______________________________________

def batch_head(x, n_head):
  b,t,c = x.shape
  x = x.reshape(b, t, n_head,-1)
  return x.transpose(1,2).reshape(b*n_head, t, -1)

def head_batch(x, n_head):
    bn, t, c = x.shape
    x = x.reshape(-1, n_head, t, c)
    return x.transpose(1, 2).reshape(-1, t, n_head * c)


class CausalMultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.n_head = config.n_head
    self.n_embd = config.n_embd
    assert self.n_embd % config.n_head == 0, "number of heads must be a multiple of n_head"
    self.head_sz = config.n_embd // config.n_head
    self.c_attn = nn.Linear(self.n_embd, 3 * self.n_embd)
    self.c_proj = nn.Linear(self.n_embd,self.n_embd)
    self.c_proj.INIT_SPECIAL_STD = 1
    # self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size)))   # -> because we switched to Flash attention

  def forward(self, x): # x -> (B, T, E)
    b,t,c = x.shape
    x = self.c_attn(x)
    q, k, v = torch.chunk(x, 3, dim=-1) # we get the qkv first before batch_head because this is the order openai used(weights optimized for it), but reversing it when training a model from scratch is fine
    q = batch_head(q, self.n_head) # (B, T, E)
    k = batch_head(k, self.n_head) # (B, T, E)
    v = batch_head(v, self.n_head) # (B, T, E)
    # wei = q @ k.transpose(-1,-2) * self.head_sz**-0.5 # Scaling, shape (B, T, T)
    # wei = wei.masked_fill(self.bias[:t,:t]==0, float('-inf'))
    # wei = F.softmax(wei, -1)
    # wei = wei @ v
    wei = F.scaled_dot_product_attention(q, k, v, is_causal=True) # is_casual=True -> causal mask is the triangular mask we built manually with torch.tril + masked_fill.
    wei = head_batch(wei, self.n_head)
    return self.c_proj(wei)


class MLP(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
    self.gelu    = nn.GELU(approximate='tanh')
    self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
    self.c_proj.INIT_SPECIAL_STD = 1
  def forward(self, x):
    return self.c_proj(self.gelu(self.c_fc(x)))



class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.attn = CausalMultiHeadAttention(config)
    self.mlp = MLP(config)
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.ln_2 = nn.LayerNorm(config.n_embd)
  def forward(self, x):
    x = x + self.attn(self.ln_1(x))
    return x + self.mlp(self.ln_2(x))


class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config
    self.transformer = nn.ModuleDict(dict(
      wpe = nn.Embedding(config.block_size, config.n_embd),
      wte = nn.Embedding(config.vocab_size, config.n_embd),
      h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
      ln_f = nn.LayerNorm(config.n_embd)
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    ## MAKE the last Linear later the same as the embedding layer
    self.lm_head.weight = self.transformer.wte.weight

    ## APPLY Initialization
    self.apply(self._initalization)

  ## Make Initialization function
  def _initalization(self, module):
      if isinstance(module, nn.Linear):
          std = 0.02
          if hasattr(module, 'INIT_SPECIAL_STD'):  std *= (2 * self.config.n_layer) ** -0.5
          nn.init.normal_(module.weight,mean=0., std=std) # torch.nn
          if module.bias is not None:  nn.init.zeros_(module.bias)
      elif isinstance(module, nn.Embedding):
          nn.init.normal_(module.weight, mean=0., std=0.02)

  def forward(self, x, targets=None): # x -> (B, Tokens)
    B, T = x.shape
    assert T <= self.config.block_size, f'The Context Exceeds the block size {T} > {self.config.block_size}'

    x = self.transformer.wpe(torch.arange(T, device=x.device)) + self.transformer.wte(x) # x -> (B, tokens, embs)\

    for layer in self.transformer.h: x = layer(x)

    x = self.transformer.ln_f(x)

    logits = self.lm_head(x)
    loss = None
    if targets is not None:  loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
    return logits, loss

  @classmethod
  def from_pretrained(cls, model_name): #  cls -> the class it self can be called -> cls()
    assert model_name in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}, f"{model_name} is not Valid" # make a set so that it is faster O(1)
    from transformers import GPT2LMHeadModel
    print(f'loading {model_name}')
    configs ={
        'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
        'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
        'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
        'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
    }[model_name]
    configs['vocab_size'] = 50257 # same in all of them
    configs['block_size'] = 1024 # same in all of them

    model_config = GPT_config(**configs)
    model = cls(model_config)
    sd = model.state_dict()
    sd_keys = sd.keys()
    sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

    model_hf = GPT2LMHeadModel.from_pretrained(model_name) # load the hugging face model
    sd_hf = model_hf.state_dict()
    sd_keys_hf = sd_hf.keys()
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
    transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight'] # openai checkpoints use a "Conv1D" module, but we only want to use a  Linear so we have to transpose these weights when we import them

    assert len(sd_keys) == len(sd_keys_hf), f'Keys mismatch : {len(sd_keys_hf)} != {len(sd_keys)}'
    for k in sd_keys_hf:
        if any(k.endswith(w) for w in transposed):
            assert sd_hf[k].shape[::-1] == sd[k].shape, f'shape mismatch Transposed if Conv-> {sd_hf[k].shape} != {sd[k].shape} at {k}'   # [::-1] reverse sequence e.g. (768, 3072) to (3072, 768)
            with torch.no_grad():
                sd[k].copy_(sd_hf[k].T)
        else:
            assert sd_hf[k].shape == sd[k].shape , f'shape mismatch {sd_hf[k].shape} != {sd[k].shape} at {k}'
            with torch.no_grad():
                sd[k].copy_(sd_hf[k])

    return model

  def optimizers_config(self, device_type):
      # Take all the requires_grad parameters - this 2 lines can be collapsed into one line but we make it in 2 for clearner steps -
      params_dict = {pn: p for pn, p in self.named_parameters()}
      params_dict = {pn: p for pn, p in params_dict.items() if p.requires_grad}

      # Decay only when <= 2 dims, dims < 2 are the 1D tensors -> biases and LayerNorm's, It doesn't make sense to decay them
      decay_params = [p for pn, p in params_dict.items() if p.dim() >= 2]
      no_decay_params = [p for pn, p in params_dict.items() if p.dim() < 2]

      # create optim group
      optim_group = [
          {'params': decay_params, 'weight_decay': self.config.weight_decay},
          {'params': no_decay_params, 'weight_decay': 0.},
      ]
      # count the params of each group
      num_decay_params = sum(p.numel() for p in decay_params)
      num_no_decay_params = sum(p.numel() for p in no_decay_params)

      # Print it
      if master_process:
          print(f'Number of the decayed Tensors : {len(decay_params)} |  Number of the decayed params : {num_decay_params:,}, ')
          print(f'Number of the decayed Tensors : {len(no_decay_params)} |  Number of no_decay_param: {num_no_decay_params:,}')

      # See if we can make fused Adam
      fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
      use_fuse = fused_available and device_type == 'cuda'
      if master_process:  print(f'Using fused AdamW: {use_fuse}')
      # Make the optim and also add the Betas and the eps that openai used in the training
      optimizer = torch.optim.AdamW(optim_group, lr=self.config.max_lr, betas=(self.config.beta1, self.config.beta2), eps=self.config.eps, fused=use_fuse)
      return optimizer


###______________________________________ MAKE A DATALOADER ________________________________

def load_tokens(filename):
    np_loaded = np.load(filename)
    np_loaded = np_loaded.astype(np.int32) # convert uint16 to int32 before cast to long, otherwise pytorch doesn't like it
    return torch.tensor(np_loaded, dtype=torch.long)




# Make DL lite
class DL_lite:
    def __init__(self, config, split):
        assert split in {'Train', 'Val'}, 'Split must be one of "Train" or "Val"'
        # store attr
        self.block_size= config.block_size
        self.process_rank = config.process_rank
        self.num_processes = config.num_processes
        self.bs = config.bs
        assert config.tot_bs_for_grad_accum % (self.bs * config.num_processes) == 0, f' Total_Batch_size: {config.tot_bs_for_grad_accum} is not divisible by Batch_size: {self.bs} * Num_processes: {self.num_processes}'
        grad_accum = bool(config.tot_bs_for_grad_accum)
        self.tot_mini_batches = config.tot_bs_for_grad_accum // (self.bs * self.block_size * self.num_processes) if grad_accum else 1
        # Loading the data
        data_root = config.data_root
        shards = os.listdir(data_root)
        shards = [s for s in shards if split in s]
        shards = sorted(shards)
        self.shards = [os.path.join(data_root, s) for s in shards]
        assert len(shards) > 0, f"no shards found for split {split}"
        if master_process: print(f'Found Shards =  {len(self.shards)} | Split = {split} | Total Batch Size = {config.tot_bs_for_grad_accum} | Grad Accum = {grad_accum} | Number Mini Batches = {self.tot_mini_batches} | Num_processes: {self.num_processes}')
        # Make a trackers
        self.reset()

    def reset(self):
        self.current_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.tr = self.bs * self.block_size * self.process_rank

    def after_batch(self):
        tokens = self.tokens[self.tr : self.tr + self.bs * self.block_size + 1 ]
        self.tr += self.bs * self.block_size * self.num_processes
        x = tokens[:-1].view(self.bs, -1)
        y = tokens[1:].view(self.bs, -1)
        if (self.bs * self.block_size * self.num_processes  + 1 + self.tr) > len(self.tokens):
            self.current_shard = (self.current_shard + 1) % len(self.shards) # So that we advance to the next shard and if we finish the shards we loop again because of -> %
            self.tokens = load_tokens(self.shards[self.current_shard])
            self.tr = self.bs * self.block_size * self.process_rank
        return x, y

###_________________________________ HELLASWAG FUNCTION _______________written in HellaSwag.py________________________________


def get_most_likely_row(tokens, mask, logits):
    shift_logits = (logits[:, :-1, :]).contiguous()
    shift_tokens = (tokens[..., 1:]).contiguous()
    shift_logits = shift_logits.view(-1, shift_logits.size(-1))
    shift_tokens = shift_tokens.view(-1)
    loss = F.cross_entropy(shift_logits, shift_tokens, reduction='none')
    reshaped_loss = loss.view(tokens.size(0), -1)
    shift_mask = mask[..., 1:].contiguous()
    masked_shift_loss = reshaped_loss * shift_mask
    sum_loss = torch.sum(masked_shift_loss, dim=-1)
    avg_loss = sum_loss / shift_mask.sum(-1)
    norm_preds = torch.argmin(avg_loss).item()

    return norm_preds

###_________________________________ LR SCHED _______________________________________________

# Creating a LR sched
def get_lr(step, config):
    min_lr = config.max_lr * config.min_lr_ratio
    if step < config.warmup_steps: return config.max_lr/config.warmup_steps * (step+1)
    if step > config.max_steps : return min_lr
    decay_ratio = (step-config.warmup_steps)/(config.max_steps-config.warmup_steps) # make a ratio between 0 and 1 that represent the current section in the cos
    assert 0 <= decay_ratio <= 1, f'There is something wrong with max steps:{config.max_steps}, step:{step}, warmup_steps:{config.warmup_steps}'
    coeff = 0.5 * (1 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (config.max_lr - min_lr)


###_____________________________________  INSTANCES  ______________________________________

# It a PyTorch function that speeds up float32 matrix multiplications on compatible NVIDIA GPUs by trading off a small amount of numerical precision for significant performance gains.
torch.set_float32_matmul_precision('high')

# config
config = GPT_config()

# make device_type
device_type = "cuda" if config.device.startswith("cuda") else "cpu" # just to use it at the autocast, etc... and make 'cuda:3' -> 'cuda', 'cuda' -> 'cuda' ,etc...

# encoder
enc = tiktoken.get_encoding(config.tokenizer)

# model
model = GPT(config)
model.to(config.device)
if config.use_compile: model = torch.compile(model) # compiles the model and makes kernel fusion for the operations
if ddp :  model = DDP(model, device_ids=[ddp_local_rank]) # Forward pass / training step → use model (the DDP wrapper) — this is what makes multi-GPU synchronization work
raw_model = model.module if ddp else model  # always contains the "raw" unwrapped model / -> Anything else (custom methods, saving checkpoints, accessing .config) → use raw_model — because DDP's wrapper doesn't expose your class's custom stuff directly

# lr getter fn
lr_getter = partial(get_lr, config=config)

# data
train_dl = DL_lite(config, 'Train')
val_dl = DL_lite(config, 'Val')

# optim
optimizer = raw_model.optimizers_config(device_type)

# make a logging file
log_dir = 'log'
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, 'log.txt')
with open(log_file, 'w') as f: # open for writing to clear the file
    pass

##_____________________________________  TRAINING  ______________________________________


# Training Loop
for step in range(config.training_steps):
    start = time.time()
    last_step = (step == config.training_steps - 1)

    # Validation
    if (step % config.val_after_step == 0 or last_step) and (config.validation) :
        model.eval()
        with torch.no_grad():
            val_accum_loss = 0.
            for _ in range(config.val_loss_accum_steps):
                x, y = val_dl.after_batch()
                x, y = x.to(config.device), y.to(config.device)
                with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
                    logits, loss = model(x, y)
                loss /= config.val_loss_accum_steps
                val_accum_loss += loss.detach()
        if ddp: dist.all_reduce(val_accum_loss, op=dist.ReduceOp.AVG)
        val_dl.reset()
        if master_process:
            print(f"Validation loss: {val_accum_loss.item():.4f}")
            with open(log_file, "a") as f: f.write(f"{step} val {val_accum_loss.item():.4f}\n")

            # Save checkpoints for the model
            if (step > 0) and (step % config.checkpoint_after_steps == 0 or last_step):
                checkpoint_path = os.path.join(log_dir, f'model_checkpoint_step_{step:05d}.pt')
                checkpoint = {
                    'model': raw_model.state_dict(),
                    'config': raw_model.config, # that is the stored config in the model object as an attr
                    'step': step,
                    'val_loss': val_accum_loss.item()
                }
                torch.save(checkpoint, checkpoint_path)


    # Model Sampling
    if step % config.val_after_step == 0 and step > 0 and config.model_sampling and (not config.use_compile):
        model.eval()
        tokens = enc.encode('I am crazy man,')
        tokens = torch.tensor(tokens, dtype=torch.long)
        tokens = tokens.repeat(config.num_sequence, 1)
        x_gen = tokens.to(config.device)

        while x_gen.shape[1] < config.max_length:
            with torch.no_grad():
                with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
                    logits, loss = model(x_gen)
                logits = logits[:, -1, :]
                probs = F.softmax(logits, dim=-1)
                topk_probs, topk_indices = torch.topk(probs, k=50, dim=-1)  # take the highest 50 probs
                ix = torch.multinomial(topk_probs, num_samples=1)  # take one sample
                ids = torch.gather(topk_indices, dim=-1, index=ix)
                x_gen = torch.cat((x_gen, ids), dim=1)

        for i in range(config.num_sequence):
            decoded = enc.decode(x_gen[i, :config.max_length].tolist())
            print(f'Rank: {config.process_rank} | Sample{i + 1}: {decoded} ')

    # HellaSwag
    # once in a while evaluate hellaswag
    if (step % config.val_after_step == 0 or last_step) and (not config.use_compile):
        num_correct_norm = 0
        num_total = 0
        for i, example in enumerate(iterate_examples("val")):
            # only process examples where i % ddp_world_size == ddp_rank
            if i % ddp_world_size != ddp_rank:  continue
            # render the example into tokens and labels
            _, tokens, mask, label = render_example(example)
            tokens = tokens.to(device)
            mask = mask.to(device)
            # get the logits
            with torch.no_grad():
                with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
                    logits, loss = model(tokens)
                pred_norm = get_most_likely_row(tokens, mask, logits)
            num_total += 1
            num_correct_norm += int(pred_norm == label)
        # reduce the stats across all processes
        if ddp:
            num_total = torch.tensor(num_total, dtype=torch.long, device=device)
            num_correct_norm = torch.tensor(num_correct_norm, dtype=torch.long, device=device)
            dist.all_reduce(num_total, op=dist.ReduceOp.SUM)
            dist.all_reduce(num_correct_norm, op=dist.ReduceOp.SUM)
            num_total = num_total.item()
            num_correct_norm = num_correct_norm.item()
        acc_norm = num_correct_norm / num_total
        if master_process:
            print(f"HellaSwag accuracy: {num_correct_norm}/{num_total}={acc_norm:.4f}")
            with open(log_file, "a") as f:  f.write(f"{step} hella {acc_norm:.4f}\n")


    # Training
    model.train()
    accum_loss = 0
    for mini_step in range(train_dl.tot_mini_batches):
        x,y = train_dl.after_batch()
        x,y = x.to(config.device), y.to(config.device)
        if ddp:  model.require_backward_grad_sync = (mini_step == train_dl.tot_mini_batches - 1) # So that we avoid unnecessary communication during backward () unless it is the last step
        with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
            logits, loss = model(x, y)
            loss /= train_dl.tot_mini_batches
            accum_loss += loss.detach()
        loss.backward()
    if ddp:  dist.all_reduce(accum_loss, op= dist.ReduceOp.AVG) # Averaging the Loss Across all the Processes
    lr = lr_getter(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.) # This Makes Normalization for the Norm of the Gradients proportionally, so that -> root(sum(grads**2)) <= 1
    optimizer.step()
    optimizer.zero_grad()
    if device_type == 'cuda' : torch.cuda.synchronize() # so that the cpu don't run the next command while the GPU still hasn't Finished

    # Printings
    end = time.time()
    time_taken = end - start
    tokens_count = train_dl.block_size * train_dl.bs * train_dl.tot_mini_batches
    if master_process:
        print(f'Step: {step:5d} | Loss: {accum_loss.item():.4f} | lr = {lr:.6f} | Norm = {norm:6f} | Time: {time_taken:.4f}sec | Token/sec: {(tokens_count / time_taken):.3f}')
        with open(log_file, 'a') as f: f.write(f"{step} train {accum_loss.item():.6f}\n")

if ddp: destroy_process_group() # Clean After the Multi-GPU Process




Writing train_gpt2.py


    FOR:             if i % ddp_world_size != ddp_rank:  continue
    
    IN HELLASWAG -> ALLOW TO CONSUME ALL THE EXAMPLES IN PARALLEL (MORE EFFICIENT)
    
    
    i=7:   7 = (1 group of 5) + 2 left over   ->  7 % 5 = 2
    i=12:  12 = (2 groups of 5) + 2 left over ->  12 % 5 = 2
    i=3:   3 = (0 groups of 5) + 3 left over  ->  3 % 5 = 3
    
                  i=0   i=1   i=2   i=3   i=4   i=5   i=6   i=7   i=8   i=9
    i % 5:         0     1     2     3     4     0     1     2     3     4
    
    rank 0:       DO    skip  skip  skip  skip   DO    skip  skip  skip  skip
    rank 1:       skip  DO    skip  skip  skip  skip   DO    skip  skip  skip
    rank 2:       skip  skip  DO    skip  skip  skip  skip   DO    skip  skip
    rank 3:       skip  skip  skip  DO    skip  skip  skip  skip   DO    skip
    rank 4:       skip  skip  skip  skip  DO    skip  skip  skip  skip   DO
    
    Every column (every example index i) has exactly one DO across all 5 ranks — meaning every example gets processed by exactly one GPU, never zero, never more than one. And the pattern repeats every 5 examples, which is exactly why i % 5 cycling through 0,1,2,3,4,0,1,2,3,4,... is what makes the round-robin distribution work for any number of GPUs — swap in world_size = 8 and you'd get the same idea, just cycling 0 through 7 instead.

In [3]:
!cd /kaggle/working

In [4]:
!rm -rf /kaggle/working/HellaSwag

In [5]:
!ls

HellaSwag.py  train_gpt2.py


In [6]:
!torchrun --standalone --nproc_per_node=2 train_gpt2.py

W0710 20:31:08.072000 93 torch/distributed/run.py:852] 
W0710 20:31:08.072000 93 torch/distributed/run.py:852] *****************************************
W0710 20:31:08.072000 93 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0710 20:31:08.072000 93 torch/distributed/run.py:852] *****************************************
[W710 20:31:08.053035385 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W710 20:31:32.018766970 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W710 20:31:32.164795162 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Found Shards =  2 | Split = Train | Total Batch Size = 131072 | Grad Accum = True | Number Mini Batches = 8 | Num_processes: 2
Found Shards =  1 | Split = 

    W0710 20:31:08.072000 93 torch/distributed/run.py:852]
    W0710 20:31:08.072000 93 torch/distributed/run.py:852] *****************************************
    W0710 20:31:08.072000 93 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
    W0710 20:31:08.072000 93 torch/distributed/run.py:852] *****************************************
    [W710 20:31:08.053035385 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
    [W710 20:31:32.018766970 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
    [W710 20:31:32.164795162 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
    Found Shards =  2 | Split = Train | Total Batch Size = 131072 | Grad Accum = True | Number Mini Batches = 8 | Num_processes: 2
    Found Shards =  1 | Split = Val | Total Batch Size = 131072 | Grad Accum = True | Number Mini Batches = 8 | Num_processes: 2
    Number of the decayed Tensors : 34 |  Number of the decayed params : 152,698,880,
    Number of the decayed Tensors : 66 |  Number of no_decay_param: 108,544
    Using fused AdamW: True
    Downloading val data from https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_val.jsonl to /kaggle/working/HellaSwag/hellaswag_val.jsonl
    Validation loss: 11.0404
    Downloading val data from https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_val.jsonl to /kaggle/working/HellaSwag/hellaswag_val.jsonl
    /kaggle/working/HellaSwag/hellaswag_val.jsonl: 11.7MiB [00:00, 46.3MiB/s]<00:00,
    /kaggle/working/HellaSwag/hellaswag_val.jsonl: 11.7MiB [00:00, 46.3MiB/s]
    HellaSwag accuracy: 2443/10042=0.2433
    Step:     0 | Loss: 11.0386 | lr = 0.000060 | Norm = 14.446857 | Time: 354.9334sec | Token/sec: 184.643
    Step:     1 | Loss: 9.8737 | lr = 0.000120 | Norm = 6.402200 | Time: 34.7454sec | Token/sec: 1886.179
    Step:     2 | Loss: 9.5159 | lr = 0.000180 | Norm = 6.219985 | Time: 35.2527sec | Token/sec: 1859.033
    Step:     3 | Loss: 9.5683 | lr = 0.000240 | Norm = 3.199221 | Time: 34.8102sec | Token/sec: 1882.664
    Step:     4 | Loss: 9.3142 | lr = 0.000300 | Norm = 2.630870 | Time: 34.6253sec | Token/sec: 1892.721
    Step:     5 | Loss: 9.0070 | lr = 0.000360 | Norm = 2.162816 | Time: 34.6062sec | Token/sec: 1893.762
    Step:     6 | Loss: 8.6080 | lr = 0.000420 | Norm = 1.806003 | Time: 34.3975sec | Token/sec: 1905.252
    Step:     7 | Loss: 8.4153 | lr = 0.000480 | Norm = 2.468205 | Time: 34.3626sec | Token/sec: 1907.192
    Step:     8 | Loss: 8.1658 | lr = 0.000540 | Norm = 2.249952 | Time: 34.6155sec | Token/sec: 1893.255
    Step:     9 | Loss: 7.9522 | lr = 0.000600 | Norm = 1.195804 | Time: 34.7908sec | Token/sec: 1883.716
    Step:    10 | Loss: 7.8773 | lr = 0.000600 | Norm = 1.404628 | Time: 34.9061sec | Token/sec: 1877.495
    Step:    11 | Loss: 7.7753 | lr = 0.000600 | Norm = 1.005864 | Time: 34.9501sec | Token/sec: 1875.133
    Step:    12 | Loss: 7.7383 | lr = 0.000600 | Norm = 0.891041 | Time: 34.9214sec | Token/sec: 1876.673
    Step:    13 | Loss: 7.7444 | lr = 0.000600 | Norm = 1.028904 | Time: 34.8598sec | Token/sec: 1879.987
    Step:    14 | Loss: 7.7445 | lr = 0.000600 | Norm = 1.091775 | Time: 34.6841sec | Token/sec: 1889.512
    Step:    15 | Loss: 7.7515 | lr = 0.000600 | Norm = 1.064513 | Time: 34.7078sec | Token/sec: 1888.223
    Step:    16 | Loss: 7.7020 | lr = 0.000600 | Norm = 0.815685 | Time: 34.6834sec | Token/sec: 1889.550
    Step:    17 | Loss: 7.7292 | lr = 0.000600 | Norm = 0.917388 | Time: 34.6652sec | Token/sec: 1890.541
    Step:    18 | Loss: 7.6876 | lr = 0.000600 | Norm = 0.850258 | Time: 34.6233sec | Token/sec: 1892.829
    Step:    19 | Loss: 7.5326 | lr = 0.000600 | Norm = 0.608874 | Time: 34.7196sec | Token/sec: 1887.579
    Step:    20 | Loss: 7.6494 | lr = 0.000600 | Norm = 0.580037 | Time: 34.9249sec | Token/sec: 1876.482
    Step:    21 | Loss: 7.5965 | lr = 0.000600 | Norm = 0.647656 | Time: 35.0823sec | Token/sec: 1868.067
    Step:    22 | Loss: 7.5425 | lr = 0.000600 | Norm = 0.642166 | Time: 35.1540sec | Token/sec: 1864.257
    Step:    23 | Loss: 7.7501 | lr = 0.000600 | Norm = 0.574836 | Time: 35.1647sec | Token/sec: 1863.686
    Step:    24 | Loss: 7.7188 | lr = 0.000600 | Norm = 0.704151 | Time: 35.2391sec | Token/sec: 1859.750
    Step:    25 | Loss: 7.4316 | lr = 0.000600 | Norm = 0.690701 | Time: 35.2898sec | Token/sec: 1857.082
    Step:    26 | Loss: 7.5410 | lr = 0.000600 | Norm = 0.604831 | Time: 35.0835sec | Token/sec: 1868.001
    Step:    27 | Loss: 7.4955 | lr = 0.000600 | Norm = 0.547059 | Time: 34.9715sec | Token/sec: 1873.985
    Step:    28 | Loss: 7.4917 | lr = 0.000599 | Norm = 0.439508 | Time: 34.9326sec | Token/sec: 1876.071
    Step:    29 | Loss: 7.5416 | lr = 0.000599 | Norm = 0.546346 | Time: 34.8543sec | Token/sec: 1880.286
    Step:    30 | Loss: 7.3925 | lr = 0.000599 | Norm = 0.532249 | Time: 35.1020sec | Token/sec: 1867.014
    Step:    31 | Loss: 7.4438 | lr = 0.000599 | Norm = 0.447462 | Time: 35.3123sec | Token/sec: 1855.896
    Step:    32 | Loss: 7.4115 | lr = 0.000599 | Norm = 0.490116 | Time: 35.3459sec | Token/sec: 1854.132
    Step:    33 | Loss: 7.3367 | lr = 0.000599 | Norm = 0.474251 | Time: 35.4194sec | Token/sec: 1850.288
    Step:    34 | Loss: 7.3385 | lr = 0.000599 | Norm = 0.499553 | Time: 35.1294sec | Token/sec: 1865.562
    Step:    35 | Loss: 7.2950 | lr = 0.000599 | Norm = 0.633205 | Time: 35.0125sec | Token/sec: 1871.787
    Step:    36 | Loss: 7.2267 | lr = 0.000599 | Norm = 0.550755 | Time: 35.0099sec | Token/sec: 1871.925
    Step:    37 | Loss: 7.2616 | lr = 0.000599 | Norm = 0.473383 | Time: 34.8437sec | Token/sec: 1880.855
    Step:    38 | Loss: 7.2310 | lr = 0.000599 | Norm = 0.849891 | Time: 34.9767sec | Token/sec: 1873.704
    Step:    39 | Loss: 7.2739 | lr = 0.000599 | Norm = 0.887362 | Time: 35.2546sec | Token/sec: 1858.937
    Step:    40 | Loss: 7.2865 | lr = 0.000598 | Norm = 0.585536 | Time: 35.3339sec | Token/sec: 1854.765
    Step:    41 | Loss: 7.2290 | lr = 0.000598 | Norm = 0.715918 | Time: 35.4490sec | Token/sec: 1848.742
    Step:    42 | Loss: 7.1902 | lr = 0.000598 | Norm = 0.658722 | Time: 35.4613sec | Token/sec: 1848.098
    Step:    43 | Loss: 7.1802 | lr = 0.000598 | Norm = 0.820537 | Time: 35.2232sec | Token/sec: 1860.593
    Step:    44 | Loss: 7.1844 | lr = 0.000598 | Norm = 0.744727 | Time: 35.0069sec | Token/sec: 1872.089
    Step:    45 | Loss: 7.0822 | lr = 0.000598 | Norm = 0.716923 | Time: 34.9944sec | Token/sec: 1872.757
    Step:    46 | Loss: 7.1069 | lr = 0.000598 | Norm = 0.710860 | Time: 34.9673sec | Token/sec: 1874.210
    Step:    47 | Loss: 7.0521 | lr = 0.000598 | Norm = 0.496118 | Time: 34.9388sec | Token/sec: 1875.734
    Step:    48 | Loss: 7.0379 | lr = 0.000598 | Norm = 0.641377 | Time: 34.8918sec | Token/sec: 1878.264
    Step:    49 | Loss: 7.0444 | lr = 0.000597 | Norm = 0.717349 | Time: 34.9857sec | Token/sec: 1873.221
    Step:    50 | Loss: 7.1427 | lr = 0.000597 | Norm = 0.516872 | Time: 35.1973sec | Token/sec: 1861.961
    Step:    51 | Loss: 7.1956 | lr = 0.000597 | Norm = 0.554855 | Time: 35.4097sec | Token/sec: 1850.793
    Step:    52 | Loss: 7.0079 | lr = 0.000597 | Norm = 0.585039 | Time: 35.4688sec | Token/sec: 1847.709
    Step:    53 | Loss: 6.9402 | lr = 0.000597 | Norm = 0.396251 | Time: 35.4472sec | Token/sec: 1848.836
    Step:    54 | Loss: 7.0536 | lr = 0.000597 | Norm = 0.533441 | Time: 35.4636sec | Token/sec: 1847.980
    Step:    55 | Loss: 7.1264 | lr = 0.000597 | Norm = 0.575277 | Time: 35.3473sec | Token/sec: 1854.062
    Step:    56 | Loss: 6.9265 | lr = 0.000596 | Norm = 0.434106 | Time: 35.0788sec | Token/sec: 1868.248
    Step:    57 | Loss: 6.8496 | lr = 0.000596 | Norm = 0.365014 | Time: 35.0549sec | Token/sec: 1869.523
    Step:    58 | Loss: 6.9101 | lr = 0.000596 | Norm = 0.484094 | Time: 35.0179sec | Token/sec: 1871.503
    Step:    59 | Loss: 6.7177 | lr = 0.000596 | Norm = 0.422045 | Time: 34.9896sec | Token/sec: 1873.015
    Step:    60 | Loss: 6.9073 | lr = 0.000596 | Norm = 0.487525 | Time: 35.2606sec | Token/sec: 1858.618
    Step:    61 | Loss: 6.8333 | lr = 0.000596 | Norm = 0.349171 | Time: 35.4168sec | Token/sec: 1850.420
    Step:    62 | Loss: 6.9055 | lr = 0.000595 | Norm = 1.021438 | Time: 35.4238sec | Token/sec: 1850.054
    Step:    63 | Loss: 6.8667 | lr = 0.000595 | Norm = 0.677078 | Time: 35.4525sec | Token/sec: 1848.560
    Step:    64 | Loss: 6.8302 | lr = 0.000595 | Norm = 0.455151 | Time: 35.3964sec | Token/sec: 1851.487
    Step:    65 | Loss: 6.9262 | lr = 0.000595 | Norm = 0.567289 | Time: 35.4995sec | Token/sec: 1846.110
    Step:    66 | Loss: 6.9565 | lr = 0.000595 | Norm = 0.690038 | Time: 35.3409sec | Token/sec: 1854.393
    Step:    67 | Loss: 6.8497 | lr = 0.000595 | Norm = 0.474518 | Time: 35.2104sec | Token/sec: 1861.268
    Step:    68 | Loss: 6.9064 | lr = 0.000594 | Norm = 0.853809 | Time: 35.1039sec | Token/sec: 1866.913
    Step:    69 | Loss: 6.7844 | lr = 0.000594 | Norm = 0.489899 | Time: 35.0511sec | Token/sec: 1869.730
    Step:    70 | Loss: 6.7985 | lr = 0.000594 | Norm = 0.578282 | Time: 34.9806sec | Token/sec: 1873.496
    Step:    71 | Loss: 6.7536 | lr = 0.000594 | Norm = 0.607975 | Time: 35.0362sec | Token/sec: 1870.525
    Step:    72 | Loss: 6.8067 | lr = 0.000594 | Norm = 0.577734 | Time: 35.1758sec | Token/sec: 1863.098
    Step:    73 | Loss: 6.9247 | lr = 0.000593 | Norm = 0.697516 | Time: 35.2711sec | Token/sec: 1858.066
    Step:    74 | Loss: 6.8277 | lr = 0.000593 | Norm = 0.541166 | Time: 35.3658sec | Token/sec: 1853.088
    Step:    75 | Loss: 7.4245 | lr = 0.000593 | Norm = 0.856970 | Time: 35.3699sec | Token/sec: 1852.877
    Step:    76 | Loss: 6.8003 | lr = 0.000593 | Norm = 0.707104 | Time: 35.4052sec | Token/sec: 1851.027
    Step:    77 | Loss: 7.6064 | lr = 0.000592 | Norm = 1.538457 | Time: 35.3659sec | Token/sec: 1853.086
    Step:    78 | Loss: 7.0737 | lr = 0.000592 | Norm = 1.077698 | Time: 35.2674sec | Token/sec: 1858.259
    Step:    79 | Loss: 6.7961 | lr = 0.000592 | Norm = 0.943513 | Time: 35.1782sec | Token/sec: 1862.973
    Step:    80 | Loss: 6.7979 | lr = 0.000592 | Norm = 0.801840 | Time: 35.1172sec | Token/sec: 1866.208
    Step:    81 | Loss: 6.8111 | lr = 0.000592 | Norm = 0.717345 | Time: 35.0554sec | Token/sec: 1869.498
    Step:    82 | Loss: 6.8030 | lr = 0.000591 | Norm = 0.659238 | Time: 35.0416sec | Token/sec: 1870.235
    Step:    83 | Loss: 6.7671 | lr = 0.000591 | Norm = 0.464673 | Time: 35.2536sec | Token/sec: 1858.988
    Step:    84 | Loss: 6.8031 | lr = 0.000591 | Norm = 0.710848 | Time: 35.4341sec | Token/sec: 1849.520
    Step:    85 | Loss: 6.7558 | lr = 0.000591 | Norm = 0.656614 | Time: 35.4623sec | Token/sec: 1848.047
    Step:    86 | Loss: 6.7088 | lr = 0.000590 | Norm = 0.544546 | Time: 35.4328sec | Token/sec: 1849.588
    Step:    87 | Loss: 6.7876 | lr = 0.000590 | Norm = 0.701588 | Time: 35.1395sec | Token/sec: 1865.023
    Step:    88 | Loss: 6.7420 | lr = 0.000590 | Norm = 0.604383 | Time: 35.0570sec | Token/sec: 1869.414
    Step:    89 | Loss: 6.6946 | lr = 0.000590 | Norm = 0.645246 | Time: 35.3618sec | Token/sec: 1853.298
    Step:    90 | Loss: 6.7087 | lr = 0.000589 | Norm = 0.675256 | Time: 35.4041sec | Token/sec: 1851.086
    Step:    91 | Loss: 6.6947 | lr = 0.000589 | Norm = 0.640384 | Time: 35.4284sec | Token/sec: 1849.813
    Step:    92 | Loss: 6.6626 | lr = 0.000589 | Norm = 0.551513 | Time: 35.2505sec | Token/sec: 1859.149
    Step:    93 | Loss: 6.6607 | lr = 0.000588 | Norm = 0.584540 | Time: 35.1374sec | Token/sec: 1865.136
    Step:    94 | Loss: 6.9436 | lr = 0.000588 | Norm = 0.849825 | Time: 35.1109sec | Token/sec: 1866.542
    Step:    95 | Loss: 6.6686 | lr = 0.000588 | Norm = 0.823048 | Time: 35.0770sec | Token/sec: 1868.349
    Step:    96 | Loss: 6.5616 | lr = 0.000588 | Norm = 0.696907 | Time: 35.0330sec | Token/sec: 1870.694
    Step:    97 | Loss: 6.6195 | lr = 0.000587 | Norm = 0.471163 | Time: 35.0465sec | Token/sec: 1869.975
    Step:    98 | Loss: 6.6938 | lr = 0.000587 | Norm = 0.745969 | Time: 35.2541sec | Token/sec: 1858.959
    Step:    99 | Loss: 6.5900 | lr = 0.000587 | Norm = 0.749514 | Time: 35.4041sec | Token/sec: 1851.087
    Validation loss: 6.6405
    Rank: 1 | Sample1: I am crazy man, and be the number that has not know at your University or a number of this is not found. The person. They have very the world (This alt need to also
    Rank: 1 | Sample2: I am crazy man, as a idea at the al.
    [d.
    It, at the same school. A, you. The US, especially to the time and ‘2 (
    Rank: 1 | Sample3: I am crazy man, such
    The United-
    -” to a result of the study, about the time into the main problem had made at a problem about a result of their local other
    Rank: 0 | Sample1: I am crazy man, and with the "If
    You:
    In a small amount of the brain year is often now the original is only their few-1-Con, to ensure the
    Rank: 0 | Sample2: I am crazy man, a future.”, but in a a same as one.<|endoftext|>[s in the heart.
    The Chinese, if the first and may give a lot.

    Rank: 0 | Sample3: I am crazy man, their body.
    -4 months can be than a other life as it is the second is a day.
    How not also not know we not make a few or a
    HellaSwag accuracy: 2425/10042=0.2415
    Step:   100 | Loss: 6.6109 | lr = 0.000586 | Norm = 0.483207 | Time: 366.3688sec | Token/sec: 178.880
    Step:   101 | Loss: 6.4707 | lr = 0.000586 | Norm = 0.619594 | Time: 35.0194sec | Token/sec: 1871.422
    Step:   102 | Loss: 6.5553 | lr = 0.000586 | Norm = 0.605145 | Time: 35.4687sec | Token/sec: 1847.712
    Step:   103 | Loss: 6.5463 | lr = 0.000586 | Norm = 0.515354 | Time: 35.3838sec | Token/sec: 1852.148
    Step:   104 | Loss: 6.6395 | lr = 0.000585 | Norm = 0.541573 | Time: 35.1109sec | Token/sec: 1866.541
    Step:   105 | Loss: 6.6143 | lr = 0.000585 | Norm = 0.524015 | Time: 35.0299sec | Token/sec: 1870.861
    Step:   106 | Loss: 6.5055 | lr = 0.000585 | Norm = 0.498638 | Time: 34.9602sec | Token/sec: 1874.589
    Step:   107 | Loss: 6.5297 | lr = 0.000584 | Norm = 0.540590 | Time: 35.3554sec | Token/sec: 1853.632
    Step:   108 | Loss: 6.5086 | lr = 0.000584 | Norm = 0.553134 | Time: 35.4618sec | Token/sec: 1848.074
    Step:   109 | Loss: 6.5013 | lr = 0.000584 | Norm = 0.424422 | Time: 35.4296sec | Token/sec: 1849.751
    Step:   110 | Loss: 6.5047 | lr = 0.000583 | Norm = 0.561435 | Time: 35.4643sec | Token/sec: 1847.945
    Step:   111 | Loss: 6.5205 | lr = 0.000583 | Norm = 0.528997 | Time: 35.3689sec | Token/sec: 1852.927
    Step:   112 | Loss: 6.5373 | lr = 0.000583 | Norm = 0.350338 | Time: 35.0595sec | Token/sec: 1869.279
    Step:   113 | Loss: 6.4974 | lr = 0.000582 | Norm = 0.547049 | Time: 34.9885sec | Token/sec: 1873.074
    Step:   114 | Loss: 6.4981 | lr = 0.000582 | Norm = 0.355339 | Time: 34.9226sec | Token/sec: 1876.608
    Step:   115 | Loss: 6.4363 | lr = 0.000582 | Norm = 0.416291 | Time: 35.1503sec | Token/sec: 1864.448
    Step:   116 | Loss: 6.4232 | lr = 0.000581 | Norm = 0.441758 | Time: 35.4362sec | Token/sec: 1849.409
    Step:   117 | Loss: 6.4461 | lr = 0.000581 | Norm = 0.360987 | Time: 35.4900sec | Token/sec: 1846.605
    Step:   118 | Loss: 6.4539 | lr = 0.000581 | Norm = 0.418904 | Time: 35.5085sec | Token/sec: 1845.644
    Step:   119 | Loss: 6.3996 | lr = 0.000580 | Norm = 0.472817 | Time: 35.0300sec | Token/sec: 1870.853
    Step:   120 | Loss: 6.5569 | lr = 0.000580 | Norm = 0.475891 | Time: 35.0428sec | Token/sec: 1870.171
    Step:   121 | Loss: 6.6288 | lr = 0.000580 | Norm = 0.461229 | Time: 35.4116sec | Token/sec: 1850.692
    Step:   122 | Loss: 6.5838 | lr = 0.000579 | Norm = 0.468106 | Time: 35.4648sec | Token/sec: 1847.917
    Step:   123 | Loss: 6.4553 | lr = 0.000579 | Norm = 0.427748 | Time: 35.3121sec | Token/sec: 1855.907
    Step:   124 | Loss: 6.4910 | lr = 0.000578 | Norm = 0.456532 | Time: 35.0703sec | Token/sec: 1868.703
    Step:   125 | Loss: 6.5190 | lr = 0.000578 | Norm = 0.394642 | Time: 35.0347sec | Token/sec: 1870.604
    Step:   126 | Loss: 6.5160 | lr = 0.000578 | Norm = 0.474846 | Time: 34.9848sec | Token/sec: 1873.270
    Step:   127 | Loss: 6.4315 | lr = 0.000577 | Norm = 0.455708 | Time: 35.1376sec | Token/sec: 1865.125
    Step:   128 | Loss: 6.4024 | lr = 0.000577 | Norm = 0.402742 | Time: 35.3087sec | Token/sec: 1856.086
    Step:   129 | Loss: 6.5238 | lr = 0.000577 | Norm = 0.544262 | Time: 35.4360sec | Token/sec: 1849.420
    Step:   130 | Loss: 6.5155 | lr = 0.000576 | Norm = 0.353327 | Time: 35.4479sec | Token/sec: 1848.796
    Step:   131 | Loss: 6.5157 | lr = 0.000576 | Norm = 0.452967 | Time: 35.4645sec | Token/sec: 1847.931
    Step:   132 | Loss: 6.5717 | lr = 0.000575 | Norm = 0.530717 | Time: 35.1960sec | Token/sec: 1862.032
    Step:   133 | Loss: 6.4586 | lr = 0.000575 | Norm = 0.520546 | Time: 35.1412sec | Token/sec: 1864.933
    Step:   134 | Loss: 6.5444 | lr = 0.000575 | Norm = 0.521002 | Time: 35.0844sec | Token/sec: 1867.951
    Step:   135 | Loss: 6.4069 | lr = 0.000574 | Norm = 0.426017 | Time: 35.1072sec | Token/sec: 1866.742
    Step:   136 | Loss: 6.4507 | lr = 0.000574 | Norm = 0.493884 | Time: 35.2518sec | Token/sec: 1859.081
    Step:   137 | Loss: 6.3337 | lr = 0.000573 | Norm = 0.466944 | Time: 35.4400sec | Token/sec: 1849.212
    Step:   138 | Loss: 6.3554 | lr = 0.000573 | Norm = 0.551751 | Time: 35.4634sec | Token/sec: 1847.989
    Step:   139 | Loss: 6.3408 | lr = 0.000572 | Norm = 0.596660 | Time: 35.4529sec | Token/sec: 1848.537
    Step:   140 | Loss: 6.3547 | lr = 0.000572 | Norm = 0.871659 | Time: 35.4387sec | Token/sec: 1849.276
    Step:   141 | Loss: 6.3337 | lr = 0.000572 | Norm = 0.473586 | Time: 35.4303sec | Token/sec: 1849.714
    Step:   142 | Loss: 6.4272 | lr = 0.000571 | Norm = 0.496929 | Time: 35.4658sec | Token/sec: 1847.866
    Step:   143 | Loss: 6.3730 | lr = 0.000571 | Norm = 0.512793 | Time: 35.2724sec | Token/sec: 1857.998
    Step:   144 | Loss: 6.3600 | lr = 0.000570 | Norm = 0.599053 | Time: 35.1530sec | Token/sec: 1864.305
    Step:   145 | Loss: 6.3445 | lr = 0.000570 | Norm = 0.439094 | Time: 34.9707sec | Token/sec: 1874.024
    Step:   146 | Loss: 6.5239 | lr = 0.000569 | Norm = 0.559860 | Time: 34.9747sec | Token/sec: 1873.810
    Step:   147 | Loss: 6.3608 | lr = 0.000569 | Norm = 0.495414 | Time: 35.2897sec | Token/sec: 1857.088
    Step:   148 | Loss: 6.3196 | lr = 0.000569 | Norm = 0.427430 | Time: 35.3327sec | Token/sec: 1854.825
    Step:   149 | Loss: 6.2138 | lr = 0.000568 | Norm = 0.442929 | Time: 35.4474sec | Token/sec: 1848.825
    Step:   150 | Loss: 6.4417 | lr = 0.000568 | Norm = 0.412038 | Time: 35.4279sec | Token/sec: 1849.842
    Step:   151 | Loss: 6.2427 | lr = 0.000567 | Norm = 0.458893 | Time: 35.4456sec | Token/sec: 1848.916
    Step:   152 | Loss: 6.2722 | lr = 0.000567 | Norm = 0.619985 | Time: 35.3086sec | Token/sec: 1856.092
    Step:   153 | Loss: 6.2069 | lr = 0.000566 | Norm = 0.581951 | Time: 35.2176sec | Token/sec: 1860.889
    Step:   154 | Loss: 6.3142 | lr = 0.000566 | Norm = 0.463687 | Time: 35.0837sec | Token/sec: 1867.990
    Step:   155 | Loss: 6.4688 | lr = 0.000565 | Norm = 0.552227 | Time: 35.0874sec | Token/sec: 1867.792
    Step:   156 | Loss: 6.2899 | lr = 0.000565 | Norm = 0.441407 | Time: 34.9417sec | Token/sec: 1875.580
    Step:   157 | Loss: 6.3923 | lr = 0.000564 | Norm = 0.387733 | Time: 35.0114sec | Token/sec: 1871.848
    Step:   158 | Loss: 6.3505 | lr = 0.000564 | Norm = 0.467001 | Time: 35.1593sec | Token/sec: 1863.974
    Step:   159 | Loss: 6.3870 | lr = 0.000564 | Norm = 0.474199 | Time: 35.3674sec | Token/sec: 1853.005
    Step:   160 | Loss: 6.2882 | lr = 0.000563 | Norm = 0.421440 | Time: 35.3710sec | Token/sec: 1852.819
    Step:   161 | Loss: 6.2748 | lr = 0.000563 | Norm = 0.438132 | Time: 35.4661sec | Token/sec: 1847.847
    Step:   162 | Loss: 6.4194 | lr = 0.000562 | Norm = 0.416045 | Time: 35.4454sec | Token/sec: 1848.930
    Step:   163 | Loss: 6.3173 | lr = 0.000562 | Norm = 0.536329 | Time: 35.4632sec | Token/sec: 1848.000
    Step:   164 | Loss: 6.2235 | lr = 0.000561 | Norm = 0.416496 | Time: 35.3670sec | Token/sec: 1853.029
    Step:   165 | Loss: 6.3505 | lr = 0.000561 | Norm = 0.432697 | Time: 35.2104sec | Token/sec: 1861.269
    Step:   166 | Loss: 6.3111 | lr = 0.000560 | Norm = 0.581355 | Time: 35.1537sec | Token/sec: 1864.270
    Step:   167 | Loss: 6.3584 | lr = 0.000560 | Norm = 0.599220 | Time: 35.0501sec | Token/sec: 1869.778
    Step:   168 | Loss: 6.2295 | lr = 0.000559 | Norm = 0.421200 | Time: 35.0233sec | Token/sec: 1871.214
    Step:   169 | Loss: 6.2715 | lr = 0.000559 | Norm = 0.533723 | Time: 35.1972sec | Token/sec: 1861.966
    Step:   170 | Loss: 6.2550 | lr = 0.000558 | Norm = 0.410181 | Time: 35.3297sec | Token/sec: 1854.982
    Step:   171 | Loss: 6.2324 | lr = 0.000558 | Norm = 0.421082 | Time: 35.4330sec | Token/sec: 1849.575
    Step:   172 | Loss: 6.2929 | lr = 0.000557 | Norm = 0.487404 | Time: 35.3902sec | Token/sec: 1851.812
    Step:   173 | Loss: 6.2606 | lr = 0.000557 | Norm = 0.389958 | Time: 35.4758sec | Token/sec: 1847.343
    Step:   174 | Loss: 6.4268 | lr = 0.000556 | Norm = 0.562370 | Time: 35.4582sec | Token/sec: 1848.263
    Step:   175 | Loss: 6.2761 | lr = 0.000555 | Norm = 1.549348 | Time: 35.4524sec | Token/sec: 1848.566
    Step:   176 | Loss: 6.3442 | lr = 0.000555 | Norm = 0.607078 | Time: 35.5078sec | Token/sec: 1845.678
    Step:   177 | Loss: 6.2948 | lr = 0.000554 | Norm = 0.593936 | Time: 35.4698sec | Token/sec: 1847.658
    Step:   178 | Loss: 6.3357 | lr = 0.000554 | Norm = 0.462338 | Time: 35.1467sec | Token/sec: 1864.640
    Step:   179 | Loss: 6.2031 | lr = 0.000553 | Norm = 0.479426 | Time: 35.0850sec | Token/sec: 1867.921
    Step:   180 | Loss: 6.3245 | lr = 0.000553 | Norm = 0.607521 | Time: 35.0913sec | Token/sec: 1867.586
    Step:   181 | Loss: 6.1940 | lr = 0.000552 | Norm = 0.480626 | Time: 35.0613sec | Token/sec: 1869.182
    Step:   182 | Loss: 6.3169 | lr = 0.000552 | Norm = 0.395772 | Time: 35.3358sec | Token/sec: 1854.664
    Step:   183 | Loss: 6.2680 | lr = 0.000551 | Norm = 0.405073 | Time: 35.4736sec | Token/sec: 1847.457
    Step:   184 | Loss: 6.1537 | lr = 0.000551 | Norm = 0.390767 | Time: 35.4703sec | Token/sec: 1847.629
    Step:   185 | Loss: 6.1815 | lr = 0.000550 | Norm = 0.515906 | Time: 35.4907sec | Token/sec: 1846.568
    Step:   186 | Loss: 6.3389 | lr = 0.000550 | Norm = 0.392370 | Time: 35.4614sec | Token/sec: 1848.094
    Step:   187 | Loss: 6.4880 | lr = 0.000549 | Norm = 0.553136 | Time: 35.3225sec | Token/sec: 1855.360
    Step:   188 | Loss: 6.3266 | lr = 0.000548 | Norm = 0.399811 | Time: 35.1161sec | Token/sec: 1866.268
    Step:   189 | Loss: 6.4449 | lr = 0.000548 | Norm = 0.514672 | Time: 35.0710sec | Token/sec: 1868.665
    Step:   190 | Loss: 6.3852 | lr = 0.000547 | Norm = 0.406013 | Time: 34.9399sec | Token/sec: 1875.678
    Step:   191 | Loss: 6.3686 | lr = 0.000547 | Norm = 0.468035 | Time: 34.9572sec | Token/sec: 1874.750
    Step:   192 | Loss: 6.4331 | lr = 0.000546 | Norm = 0.415964 | Time: 35.0441sec | Token/sec: 1870.103
    Step:   193 | Loss: 6.3044 | lr = 0.000546 | Norm = 0.401760 | Time: 35.3003sec | Token/sec: 1856.530
    Step:   194 | Loss: 6.4387 | lr = 0.000545 | Norm = 0.385586 | Time: 35.3907sec | Token/sec: 1851.788
    Step:   195 | Loss: 6.3990 | lr = 0.000544 | Norm = 0.413272 | Time: 35.4336sec | Token/sec: 1849.542
    Step:   196 | Loss: 6.2878 | lr = 0.000544 | Norm = 0.394781 | Time: 35.4439sec | Token/sec: 1849.004
    Step:   197 | Loss: 6.3015 | lr = 0.000543 | Norm = 0.385089 | Time: 35.5164sec | Token/sec: 1845.235
    Step:   198 | Loss: 6.2796 | lr = 0.000543 | Norm = 0.402382 | Time: 35.3009sec | Token/sec: 1856.497
    Step:   199 | Loss: 6.3787 | lr = 0.000542 | Norm = 0.386568 | Time: 35.2482sec | Token/sec: 1859.274
    Validation loss: 6.2976
    Rank: 1 | Sample1: I am crazy man, and some of the need you be saved the way that of it should it by the way but they shall want to help. I am with you should not be a very serious
    Rank: 1 | Sample2: I am crazy man, for men, the end of this would have the second. One is all of these to the time that the word, and the other place to provide it was still the next
    Rank: 1 | Sample3: I am crazy man, but it took the same in some of one year is now at us about the day that the same, at the world in most of the next year, and has not one
    Rank: 0 | Sample1: I am crazy man, but, as one, when they said he is all the most serious for another and to the
    B), not on a very small school time. (p. And
    Rank: 0 | Sample2: I am crazy man, that will be at us just one is they have been not, with that.
    There is a lot of an additional a little one but your person! Let not, it
    Rank: 0 | Sample3: I am crazy man, then is as one to be used.’s one you have seen like you and it might be about an the other longer in the more likely to think, not take
    HellaSwag accuracy: 2457/10042=0.2447
    Step:   200 | Loss: 6.3990 | lr = 0.000542 | Norm = 0.406722 | Time: 367.6167sec | Token/sec: 178.273
    Step:   201 | Loss: 6.3011 | lr = 0.000541 | Norm = 0.429037 | Time: 34.9877sec | Token/sec: 1873.118
    Step:   202 | Loss: 6.3933 | lr = 0.000540 | Norm = 0.362642 | Time: 35.4570sec | Token/sec: 1848.325
    Step:   203 | Loss: 6.3656 | lr = 0.000540 | Norm = 0.467749 | Time: 35.0085sec | Token/sec: 1872.004
    Step:   204 | Loss: 6.3467 | lr = 0.000539 | Norm = 0.366146 | Time: 35.3026sec | Token/sec: 1856.405
    Step:   205 | Loss: 6.3372 | lr = 0.000539 | Norm = 0.434041 | Time: 35.4808sec | Token/sec: 1847.082
    Step:   206 | Loss: 6.3596 | lr = 0.000538 | Norm = 0.391090 | Time: 35.0489sec | Token/sec: 1869.842
    Step:   207 | Loss: 6.2690 | lr = 0.000537 | Norm = 0.388726 | Time: 35.0403sec | Token/sec: 1870.306
    Step:   208 | Loss: 6.3189 | lr = 0.000537 | Norm = 0.435755 | Time: 35.2351sec | Token/sec: 1859.963
    Step:   209 | Loss: 6.3787 | lr = 0.000536 | Norm = 0.358314 | Time: 35.4656sec | Token/sec: 1847.876
    Step:   210 | Loss: 6.3494 | lr = 0.000535 | Norm = 0.409859 | Time: 35.4295sec | Token/sec: 1849.756
    Step:   211 | Loss: 6.2541 | lr = 0.000535 | Norm = 0.400286 | Time: 35.4860sec | Token/sec: 1846.815
    Step:   212 | Loss: 6.3148 | lr = 0.000534 | Norm = 0.379095 | Time: 35.4464sec | Token/sec: 1848.874
    Step:   213 | Loss: 6.2572 | lr = 0.000534 | Norm = 0.387159 | Time: 35.4691sec | Token/sec: 1847.695
    Step:   214 | Loss: 6.2799 | lr = 0.000533 | Norm = 0.445800 | Time: 35.5155sec | Token/sec: 1845.278
    Step:   215 | Loss: 6.1920 | lr = 0.000532 | Norm = 0.467898 | Time: 35.4729sec | Token/sec: 1847.493
    Step:   216 | Loss: 6.1954 | lr = 0.000532 | Norm = 0.387758 | Time: 35.1417sec | Token/sec: 1864.909
    Step:   217 | Loss: 6.2759 | lr = 0.000531 | Norm = 0.297753 | Time: 34.9907sec | Token/sec: 1872.954
    Step:   218 | Loss: 6.1787 | lr = 0.000530 | Norm = 0.390886 | Time: 34.9954sec | Token/sec: 1872.705
    Step:   219 | Loss: 6.3992 | lr = 0.000530 | Norm = 0.344130 | Time: 35.0747sec | Token/sec: 1868.472
    Step:   220 | Loss: 6.3098 | lr = 0.000529 | Norm = 0.356240 | Time: 35.2369sec | Token/sec: 1859.871
    Step:   221 | Loss: 6.2334 | lr = 0.000529 | Norm = 0.623819 | Time: 35.4104sec | Token/sec: 1850.755
    Step:   222 | Loss: 6.2522 | lr = 0.000528 | Norm = 0.584781 | Time: 35.4601sec | Token/sec: 1848.162
    Step:   223 | Loss: 6.2753 | lr = 0.000527 | Norm = 0.340960 | Time: 35.4549sec | Token/sec: 1848.432
    Step:   224 | Loss: 6.1976 | lr = 0.000527 | Norm = 0.489783 | Time: 35.4593sec | Token/sec: 1848.205
    Step:   225 | Loss: 6.2754 | lr = 0.000526 | Norm = 0.442943 | Time: 35.4867sec | Token/sec: 1846.774
    Step:   226 | Loss: 6.2996 | lr = 0.000525 | Norm = 0.434579 | Time: 35.2205sec | Token/sec: 1860.735
    Step:   227 | Loss: 6.2711 | lr = 0.000525 | Norm = 0.408632 | Time: 35.1790sec | Token/sec: 1862.927
    Step:   228 | Loss: 6.2700 | lr = 0.000524 | Norm = 0.483333 | Time: 35.0840sec | Token/sec: 1867.975
    Step:   229 | Loss: 6.3078 | lr = 0.000523 | Norm = 0.485851 | Time: 34.9970sec | Token/sec: 1872.619
    Step:   230 | Loss: 6.2835 | lr = 0.000523 | Norm = 0.430942 | Time: 35.2755sec | Token/sec: 1857.832
    Step:   231 | Loss: 6.2199 | lr = 0.000522 | Norm = 0.415748 | Time: 35.4310sec | Token/sec: 1849.682
    Step:   232 | Loss: 6.1844 | lr = 0.000521 | Norm = 0.362567 | Time: 35.4695sec | Token/sec: 1847.670
    Step:   233 | Loss: 6.2658 | lr = 0.000521 | Norm = 0.353444 | Time: 35.4717sec | Token/sec: 1847.555
    Step:   234 | Loss: 6.1040 | lr = 0.000520 | Norm = 0.371035 | Time: 35.4845sec | Token/sec: 1846.889
    Step:   235 | Loss: 6.2526 | lr = 0.000519 | Norm = 0.400108 | Time: 35.3977sec | Token/sec: 1851.418
    Step:   236 | Loss: 6.1298 | lr = 0.000519 | Norm = 0.345526 | Time: 35.2029sec | Token/sec: 1861.665
    Step:   237 | Loss: 6.2214 | lr = 0.000518 | Norm = 0.351329 | Time: 35.0980sec | Token/sec: 1867.227
    Step:   238 | Loss: 6.1890 | lr = 0.000517 | Norm = 0.353802 | Time: 34.9932sec | Token/sec: 1872.821
    Step:   239 | Loss: 6.1909 | lr = 0.000516 | Norm = 0.330973 | Time: 35.0305sec | Token/sec: 1870.825
    Step:   240 | Loss: 6.2393 | lr = 0.000516 | Norm = 0.327631 | Time: 35.0880sec | Token/sec: 1867.761
    Step:   241 | Loss: 6.2209 | lr = 0.000515 | Norm = 0.400222 | Time: 35.1876sec | Token/sec: 1862.472
    Step:   242 | Loss: 6.1860 | lr = 0.000514 | Norm = 0.335262 | Time: 35.3237sec | Token/sec: 1855.299
    Step:   243 | Loss: 6.1121 | lr = 0.000514 | Norm = 0.331511 | Time: 35.4589sec | Token/sec: 1848.225
    Step:   244 | Loss: 6.2313 | lr = 0.000513 | Norm = 0.347446 | Time: 35.4525sec | Token/sec: 1848.557
    Step:   245 | Loss: 6.0233 | lr = 0.000512 | Norm = 0.506636 | Time: 35.4686sec | Token/sec: 1847.720
    Step:   246 | Loss: 6.2945 | lr = 0.000512 | Norm = 0.460004 | Time: 35.4535sec | Token/sec: 1848.503
    Step:   247 | Loss: 6.1931 | lr = 0.000511 | Norm = 0.493638 | Time: 35.3130sec | Token/sec: 1855.859
    Step:   248 | Loss: 6.1348 | lr = 0.000510 | Norm = 0.404727 | Time: 35.1930sec | Token/sec: 1862.187
    Step:   249 | Loss: 6.1200 | lr = 0.000509 | Norm = 0.401721 | Time: 35.2077sec | Token/sec: 1861.411
    Step:   250 | Loss: 6.1789 | lr = 0.000509 | Norm = 0.357828 | Time: 35.1369sec | Token/sec: 1865.163
    Step:   251 | Loss: 6.1058 | lr = 0.000508 | Norm = 0.364829 | Time: 35.0711sec | Token/sec: 1868.662
    Step:   252 | Loss: 6.2404 | lr = 0.000507 | Norm = 0.397132 | Time: 35.0520sec | Token/sec: 1869.680
    Step:   253 | Loss: 6.1254 | lr = 0.000507 | Norm = 0.337849 | Time: 35.0510sec | Token/sec: 1869.735
    Step:   254 | Loss: 6.0613 | lr = 0.000506 | Norm = 0.387618 | Time: 35.0802sec | Token/sec: 1868.178
    Step:   255 | Loss: 6.0812 | lr = 0.000505 | Norm = 0.399548 | Time: 35.2180sec | Token/sec: 1860.868
    Step:   256 | Loss: 6.1389 | lr = 0.000504 | Norm = 0.392821 | Time: 35.3942sec | Token/sec: 1851.602
    Step:   257 | Loss: 6.1375 | lr = 0.000504 | Norm = 0.336645 | Time: 35.4724sec | Token/sec: 1847.523
    Step:   258 | Loss: 6.0686 | lr = 0.000503 | Norm = 0.364824 | Time: 35.4488sec | Token/sec: 1848.751
    Step:   259 | Loss: 6.1136 | lr = 0.000502 | Norm = 0.384476 | Time: 35.4679sec | Token/sec: 1847.757
    Step:   260 | Loss: 6.1629 | lr = 0.000502 | Norm = 0.345456 | Time: 35.4524sec | Token/sec: 1848.566
    Step:   261 | Loss: 6.1366 | lr = 0.000501 | Norm = 0.357865 | Time: 35.2326sec | Token/sec: 1860.095
    Step:   262 | Loss: 6.1881 | lr = 0.000500 | Norm = 0.435134 | Time: 35.0717sec | Token/sec: 1868.628
    Step:   263 | Loss: 6.1079 | lr = 0.000499 | Norm = 0.397676 | Time: 35.0871sec | Token/sec: 1867.811
    Step:   264 | Loss: 6.1166 | lr = 0.000499 | Norm = 0.510677 | Time: 35.1073sec | Token/sec: 1866.733
    Step:   265 | Loss: 6.1168 | lr = 0.000498 | Norm = 0.518914 | Time: 35.4014sec | Token/sec: 1851.228
    Step:   266 | Loss: 6.0547 | lr = 0.000497 | Norm = 0.406106 | Time: 35.5036sec | Token/sec: 1845.897
    Step:   267 | Loss: 6.1118 | lr = 0.000496 | Norm = 0.516239 | Time: 35.4480sec | Token/sec: 1848.794
    Step:   268 | Loss: 6.1804 | lr = 0.000496 | Norm = 0.484064 | Time: 35.4432sec | Token/sec: 1849.043
    Step:   269 | Loss: 6.1347 | lr = 0.000495 | Norm = 0.466019 | Time: 35.4729sec | Token/sec: 1847.497
    Step:   270 | Loss: 6.2016 | lr = 0.000494 | Norm = 0.378236 | Time: 35.3053sec | Token/sec: 1856.263
    Step:   271 | Loss: 6.0615 | lr = 0.000493 | Norm = 0.428119 | Time: 35.2309sec | Token/sec: 1860.184
    Step:   272 | Loss: 6.1009 | lr = 0.000493 | Norm = 0.468571 | Time: 35.1357sec | Token/sec: 1865.225
    Step:   273 | Loss: 6.1420 | lr = 0.000492 | Norm = 0.443423 | Time: 35.0912sec | Token/sec: 1867.591
    Step:   274 | Loss: 6.1115 | lr = 0.000491 | Norm = 0.462696 | Time: 35.0506sec | Token/sec: 1869.755
    Step:   275 | Loss: 6.0895 | lr = 0.000490 | Norm = 0.387541 | Time: 35.2714sec | Token/sec: 1858.052
    Step:   276 | Loss: 6.0632 | lr = 0.000489 | Norm = 0.335604 | Time: 35.4352sec | Token/sec: 1849.463
    Step:   277 | Loss: 5.9311 | lr = 0.000489 | Norm = 0.356980 | Time: 35.4283sec | Token/sec: 1849.820
    Step:   278 | Loss: 6.0647 | lr = 0.000488 | Norm = 0.383519 | Time: 35.4771sec | Token/sec: 1847.274
    Step:   279 | Loss: 5.9675 | lr = 0.000487 | Norm = 0.326585 | Time: 35.2114sec | Token/sec: 1861.217
    Step:   280 | Loss: 6.1212 | lr = 0.000486 | Norm = 0.477401 | Time: 35.0365sec | Token/sec: 1870.509
    Step:   281 | Loss: 6.0038 | lr = 0.000486 | Norm = 0.370358 | Time: 35.0528sec | Token/sec: 1869.634
    Step:   282 | Loss: 5.9892 | lr = 0.000485 | Norm = 0.402913 | Time: 35.1161sec | Token/sec: 1866.266
    Step:   283 | Loss: 5.9371 | lr = 0.000484 | Norm = 0.356488 | Time: 35.0201sec | Token/sec: 1871.380
    Step:   284 | Loss: 6.0204 | lr = 0.000483 | Norm = 0.378909 | Time: 35.3143sec | Token/sec: 1855.792
    Step:   285 | Loss: 6.0491 | lr = 0.000482 | Norm = 0.394250 | Time: 35.4567sec | Token/sec: 1848.340
    Step:   286 | Loss: 6.0126 | lr = 0.000482 | Norm = 0.395156 | Time: 35.4753sec | Token/sec: 1847.372
    Step:   287 | Loss: 6.0513 | lr = 0.000481 | Norm = 0.372577 | Time: 35.5441sec | Token/sec: 1843.795
    Step:   288 | Loss: 6.0003 | lr = 0.000480 | Norm = 0.438001 | Time: 35.4730sec | Token/sec: 1847.490
    Step:   289 | Loss: 6.0131 | lr = 0.000479 | Norm = 0.364359 | Time: 35.4451sec | Token/sec: 1848.945
    Step:   290 | Loss: 6.0254 | lr = 0.000479 | Norm = 0.404819 | Time: 35.3314sec | Token/sec: 1854.892
    Step:   291 | Loss: 5.9190 | lr = 0.000478 | Norm = 0.380329 | Time: 35.2689sec | Token/sec: 1858.183
    Step:   292 | Loss: 5.9495 | lr = 0.000477 | Norm = 0.369047 | Time: 35.2595sec | Token/sec: 1858.675
    Step:   293 | Loss: 6.0114 | lr = 0.000476 | Norm = 0.356552 | Time: 35.4115sec | Token/sec: 1850.697
    Step:   294 | Loss: 6.0093 | lr = 0.000475 | Norm = 0.421766 | Time: 35.4845sec | Token/sec: 1846.893
    Step:   295 | Loss: 6.0181 | lr = 0.000475 | Norm = 0.415981 | Time: 35.5158sec | Token/sec: 1845.264
    Step:   296 | Loss: 5.9135 | lr = 0.000474 | Norm = 0.346475 | Time: 35.2166sec | Token/sec: 1860.942
    Step:   297 | Loss: 5.8993 | lr = 0.000473 | Norm = 0.346036 | Time: 35.1010sec | Token/sec: 1867.067
    Step:   298 | Loss: 5.9619 | lr = 0.000472 | Norm = 0.359475 | Time: 35.0095sec | Token/sec: 1871.950
    Step:   299 | Loss: 5.9485 | lr = 0.000471 | Norm = 0.386467 | Time: 34.9538sec | Token/sec: 1874.934
    Validation loss: 5.9975
    Rank: 1 | Sample1: I am crazy man, while, so you was very hard to understand for their own.
    Now that had a large, that they can get a thing to be
    They are being so a full
    Rank: 1 | Sample2: I am crazy man, the name, or so, you. That makes your best of your work in God, a good work is not a great. In general. So you can find the next
    Rank: 1 | Sample3: I am crazy man, would think in her own own own friends. You must go that can leave on you know so-off to remember your child, and she don�t love our ideas!
    Rank: 0 | Sample1: I am crazy man, it was being also just what was not.’s got.’t say,’s time and what she would see a “he did with the
    Rank: 0 | Sample2: I am crazy man, and for the same as the world for the Lord of all the Bible's God; he do they do’t a person.
    The story is the text for others
    Rank: 0 | Sample3: I am crazy man, for all the day.
    For them.
    One of what are made a time for the one of one of an�s will never be used to think about 1-
    HellaSwag accuracy: 2434/10042=0.2424
    Step:   300 | Loss: 5.9345 | lr = 0.000470 | Norm = 0.423578 | Time: 366.5015sec | Token/sec: 178.815
    Step:   301 | Loss: 6.0601 | lr = 0.000470 | Norm = 0.426778 | Time: 35.3506sec | Token/sec: 1853.887
    Step:   302 | Loss: 5.9239 | lr = 0.000469 | Norm = 0.455283 | Time: 35.3460sec | Token/sec: 1854.127
    Step:   303 | Loss: 5.9538 | lr = 0.000468 | Norm = 0.424667 | Time: 35.0133sec | Token/sec: 1871.746
    Step:   304 | Loss: 5.8923 | lr = 0.000467 | Norm = 0.426591 | Time: 35.0944sec | Token/sec: 1867.422
    Step:   305 | Loss: 5.8828 | lr = 0.000466 | Norm = 0.440576 | Time: 35.0730sec | Token/sec: 1868.562
    Step:   306 | Loss: 5.8934 | lr = 0.000466 | Norm = 0.401319 | Time: 35.0765sec | Token/sec: 1868.373
    Step:   307 | Loss: 6.0197 | lr = 0.000465 | Norm = 0.598536 | Time: 35.0382sec | Token/sec: 1870.418
    Step:   308 | Loss: 5.9034 | lr = 0.000464 | Norm = 0.482822 | Time: 35.0334sec | Token/sec: 1870.672
    Step:   309 | Loss: 5.9337 | lr = 0.000463 | Norm = 0.447988 | Time: 35.2327sec | Token/sec: 1860.088
    Step:   310 | Loss: 5.9948 | lr = 0.000462 | Norm = 0.491553 | Time: 35.3368sec | Token/sec: 1854.611
    Step:   311 | Loss: 6.0354 | lr = 0.000461 | Norm = 0.462930 | Time: 35.4594sec | Token/sec: 1848.199
    Step:   312 | Loss: 6.0449 | lr = 0.000461 | Norm = 0.408729 | Time: 35.4585sec | Token/sec: 1848.246
    Step:   313 | Loss: 5.9015 | lr = 0.000460 | Norm = 0.402050 | Time: 35.4768sec | Token/sec: 1847.291
    Step:   314 | Loss: 5.8944 | lr = 0.000459 | Norm = 0.368379 | Time: 35.4980sec | Token/sec: 1846.187
    Step:   315 | Loss: 5.8979 | lr = 0.000458 | Norm = 0.359345 | Time: 35.4161sec | Token/sec: 1850.459
    Step:   316 | Loss: 5.9244 | lr = 0.000457 | Norm = 0.329556 | Time: 34.9500sec | Token/sec: 1875.136
    Step:   317 | Loss: 5.9708 | lr = 0.000456 | Norm = 0.359280 | Time: 35.0343sec | Token/sec: 1870.622
    Step:   318 | Loss: 5.8534 | lr = 0.000456 | Norm = 0.397008 | Time: 35.3119sec | Token/sec: 1855.916
    Step:   319 | Loss: 5.8096 | lr = 0.000455 | Norm = 0.402416 | Time: 35.4328sec | Token/sec: 1849.586
    Step:   320 | Loss: 5.9022 | lr = 0.000454 | Norm = 0.406848 | Time: 35.4623sec | Token/sec: 1848.049
    Step:   321 | Loss: 5.9286 | lr = 0.000453 | Norm = 0.373664 | Time: 35.4152sec | Token/sec: 1850.506
    Step:   322 | Loss: 5.8253 | lr = 0.000452 | Norm = 0.446519 | Time: 35.1899sec | Token/sec: 1862.350
    Step:   323 | Loss: 5.9163 | lr = 0.000451 | Norm = 0.398152 | Time: 35.3716sec | Token/sec: 1852.788
    Step:   324 | Loss: 5.8069 | lr = 0.000450 | Norm = 0.367810 | Time: 35.3904sec | Token/sec: 1851.804
    Step:   325 | Loss: 5.8264 | lr = 0.000450 | Norm = 0.381941 | Time: 35.2375sec | Token/sec: 1859.836
    Step:   326 | Loss: 5.8993 | lr = 0.000449 | Norm = 0.335784 | Time: 35.1758sec | Token/sec: 1863.097
    Step:   327 | Loss: 5.9132 | lr = 0.000448 | Norm = 0.373768 | Time: 34.9772sec | Token/sec: 1873.680
    Step:   328 | Loss: 5.8844 | lr = 0.000447 | Norm = 0.398757 | Time: 35.2152sec | Token/sec: 1861.014
    Step:   329 | Loss: 5.8299 | lr = 0.000446 | Norm = 0.364904 | Time: 35.4327sec | Token/sec: 1849.589
    Step:   330 | Loss: 5.8348 | lr = 0.000445 | Norm = 0.383074 | Time: 35.4593sec | Token/sec: 1848.204
    Step:   331 | Loss: 5.8161 | lr = 0.000444 | Norm = 0.503795 | Time: 35.5164sec | Token/sec: 1845.231
    Step:   332 | Loss: 5.7716 | lr = 0.000444 | Norm = 0.595706 | Time: 35.3213sec | Token/sec: 1855.425
    Step:   333 | Loss: 5.8564 | lr = 0.000443 | Norm = 0.459133 | Time: 35.0231sec | Token/sec: 1871.223
    Step:   334 | Loss: 5.7741 | lr = 0.000442 | Norm = 0.445811 | Time: 34.9779sec | Token/sec: 1873.638
    Step:   335 | Loss: 5.7480 | lr = 0.000441 | Norm = 0.478494 | Time: 35.1408sec | Token/sec: 1864.956
    Step:   336 | Loss: 5.8106 | lr = 0.000440 | Norm = 0.432743 | Time: 35.4340sec | Token/sec: 1849.524
    Step:   337 | Loss: 5.7788 | lr = 0.000439 | Norm = 0.418158 | Time: 35.4610sec | Token/sec: 1848.116
    Step:   338 | Loss: 5.7681 | lr = 0.000438 | Norm = 0.358891 | Time: 35.4912sec | Token/sec: 1846.543
    Step:   339 | Loss: 5.8449 | lr = 0.000437 | Norm = 0.342840 | Time: 35.4249sec | Token/sec: 1849.998
    Step:   340 | Loss: 5.8127 | lr = 0.000437 | Norm = 0.406233 | Time: 35.1463sec | Token/sec: 1864.664
    Step:   341 | Loss: 5.7546 | lr = 0.000436 | Norm = 0.372746 | Time: 35.1610sec | Token/sec: 1863.882
    Step:   342 | Loss: 5.6913 | lr = 0.000435 | Norm = 0.358280 | Time: 35.1508sec | Token/sec: 1864.422
    Step:   343 | Loss: 5.7803 | lr = 0.000434 | Norm = 0.356493 | Time: 35.3007sec | Token/sec: 1856.508
    Step:   344 | Loss: 5.7391 | lr = 0.000433 | Norm = 0.378976 | Time: 35.4487sec | Token/sec: 1848.757
    Step:   345 | Loss: 5.7093 | lr = 0.000432 | Norm = 0.385853 | Time: 35.4514sec | Token/sec: 1848.613
    Step:   346 | Loss: 5.7837 | lr = 0.000431 | Norm = 0.446989 | Time: 35.4742sec | Token/sec: 1847.427
    Step:   347 | Loss: 5.7653 | lr = 0.000430 | Norm = 0.395996 | Time: 35.4529sec | Token/sec: 1848.536
    Step:   348 | Loss: 5.7820 | lr = 0.000430 | Norm = 0.370794 | Time: 35.4543sec | Token/sec: 1848.463
    Step:   349 | Loss: 5.8069 | lr = 0.000429 | Norm = 0.356054 | Time: 35.2669sec | Token/sec: 1858.286
    Step:   350 | Loss: 5.8086 | lr = 0.000428 | Norm = 0.411737 | Time: 35.0989sec | Token/sec: 1867.180
    Step:   351 | Loss: 5.8033 | lr = 0.000427 | Norm = 0.406656 | Time: 35.0659sec | Token/sec: 1868.936
    Step:   352 | Loss: 5.8146 | lr = 0.000426 | Norm = 0.503532 | Time: 35.1398sec | Token/sec: 1865.008
    Step:   353 | Loss: 5.7757 | lr = 0.000425 | Norm = 0.497729 | Time: 34.9779sec | Token/sec: 1873.640
    Step:   354 | Loss: 5.7640 | lr = 0.000424 | Norm = 0.475930 | Time: 35.1699sec | Token/sec: 1863.412
    Step:   355 | Loss: 5.7449 | lr = 0.000423 | Norm = 0.418371 | Time: 35.3778sec | Token/sec: 1852.459
    Step:   356 | Loss: 5.7974 | lr = 0.000422 | Norm = 0.447319 | Time: 35.4388sec | Token/sec: 1849.273
    Step:   357 | Loss: 5.7966 | lr = 0.000422 | Norm = 0.429540 | Time: 35.4353sec | Token/sec: 1849.453
    Step:   358 | Loss: 5.7283 | lr = 0.000421 | Norm = 0.417241 | Time: 35.4877sec | Token/sec: 1846.726
    Step:   359 | Loss: 5.8980 | lr = 0.000420 | Norm = 0.590697 | Time: 35.5087sec | Token/sec: 1845.632
    Step:   360 | Loss: 5.7680 | lr = 0.000419 | Norm = 0.528756 | Time: 35.3694sec | Token/sec: 1852.901
    Step:   361 | Loss: 5.7713 | lr = 0.000418 | Norm = 0.452948 | Time: 35.2911sec | Token/sec: 1857.012
    Step:   362 | Loss: 5.7227 | lr = 0.000417 | Norm = 0.500451 | Time: 35.2173sec | Token/sec: 1860.902
    Step:   363 | Loss: 5.7373 | lr = 0.000416 | Norm = 0.433846 | Time: 35.1373sec | Token/sec: 1865.139
    Step:   364 | Loss: 5.7527 | lr = 0.000415 | Norm = 0.522880 | Time: 35.2544sec | Token/sec: 1858.943
    Step:   365 | Loss: 5.7364 | lr = 0.000414 | Norm = 0.432189 | Time: 35.4301sec | Token/sec: 1849.727
    Step:   366 | Loss: 5.8289 | lr = 0.000413 | Norm = 0.427471 | Time: 35.5110sec | Token/sec: 1845.511
    Step:   367 | Loss: 5.6900 | lr = 0.000413 | Norm = 0.449768 | Time: 35.5031sec | Token/sec: 1845.922
    Step:   368 | Loss: 5.8153 | lr = 0.000412 | Norm = 0.428440 | Time: 34.9949sec | Token/sec: 1872.729
    Step:   369 | Loss: 5.7097 | lr = 0.000411 | Norm = 0.401846 | Time: 35.4112sec | Token/sec: 1850.711
    Step:   370 | Loss: 5.8668 | lr = 0.000410 | Norm = 0.429848 | Time: 35.5117sec | Token/sec: 1845.479
    Step:   371 | Loss: 5.8168 | lr = 0.000409 | Norm = 0.394929 | Time: 35.5325sec | Token/sec: 1844.398
    Step:   372 | Loss: 5.8820 | lr = 0.000408 | Norm = 0.416767 | Time: 34.9396sec | Token/sec: 1875.693
    Step:   373 | Loss: 5.8093 | lr = 0.000407 | Norm = 0.414774 | Time: 35.4118sec | Token/sec: 1850.684
    Step:   374 | Loss: 5.8512 | lr = 0.000406 | Norm = 0.449449 | Time: 35.3538sec | Token/sec: 1853.721
    Step:   375 | Loss: 5.9444 | lr = 0.000405 | Norm = 0.475706 | Time: 35.0285sec | Token/sec: 1870.933
    Step:   376 | Loss: 5.9714 | lr = 0.000404 | Norm = 0.401369 | Time: 35.5108sec | Token/sec: 1845.522
    Step:   377 | Loss: 5.8640 | lr = 0.000403 | Norm = 0.392888 | Time: 35.0459sec | Token/sec: 1870.005
    Step:   378 | Loss: 5.8304 | lr = 0.000402 | Norm = 0.492841 | Time: 35.3779sec | Token/sec: 1852.456
    Step:   379 | Loss: 5.9393 | lr = 0.000402 | Norm = 0.384307 | Time: 35.4910sec | Token/sec: 1846.551
    Step:   380 | Loss: 5.9610 | lr = 0.000401 | Norm = 0.424973 | Time: 35.3347sec | Token/sec: 1854.720
    Step:   381 | Loss: 5.8776 | lr = 0.000400 | Norm = 0.456592 | Time: 35.2318sec | Token/sec: 1860.138
    Step:   382 | Loss: 5.8438 | lr = 0.000399 | Norm = 0.440173 | Time: 35.0564sec | Token/sec: 1869.445
    Step:   383 | Loss: 5.8561 | lr = 0.000398 | Norm = 0.459526 | Time: 35.1039sec | Token/sec: 1866.916
    Step:   384 | Loss: 5.8043 | lr = 0.000397 | Norm = 0.463194 | Time: 35.3686sec | Token/sec: 1852.941
    Step:   385 | Loss: 5.8023 | lr = 0.000396 | Norm = 0.396469 | Time: 35.4316sec | Token/sec: 1849.649
    Step:   386 | Loss: 5.9226 | lr = 0.000395 | Norm = 0.638547 | Time: 35.5084sec | Token/sec: 1845.645
    Step:   387 | Loss: 5.8476 | lr = 0.000394 | Norm = 0.610356 | Time: 35.1268sec | Token/sec: 1865.700
    Step:   388 | Loss: 5.7372 | lr = 0.000393 | Norm = 0.494653 | Time: 35.0133sec | Token/sec: 1871.747
    Step:   389 | Loss: 5.7981 | lr = 0.000392 | Norm = 0.458304 | Time: 34.9741sec | Token/sec: 1873.846
    Step:   390 | Loss: 5.8578 | lr = 0.000391 | Norm = 0.474561 | Time: 35.1576sec | Token/sec: 1864.062
    Step:   391 | Loss: 5.7771 | lr = 0.000390 | Norm = 0.467434 | Time: 35.4156sec | Token/sec: 1850.485
    Step:   392 | Loss: 5.9292 | lr = 0.000390 | Norm = 0.630998 | Time: 35.4333sec | Token/sec: 1849.558
    Step:   393 | Loss: 5.8309 | lr = 0.000389 | Norm = 0.555055 | Time: 35.4891sec | Token/sec: 1846.650
    Step:   394 | Loss: 5.8070 | lr = 0.000388 | Norm = 0.623359 | Time: 35.5004sec | Token/sec: 1846.062
    Step:   395 | Loss: 5.8300 | lr = 0.000387 | Norm = 0.482936 | Time: 35.5192sec | Token/sec: 1845.089
    Step:   396 | Loss: 5.7520 | lr = 0.000386 | Norm = 0.532944 | Time: 35.4615sec | Token/sec: 1848.090
    Step:   397 | Loss: 5.8781 | lr = 0.000385 | Norm = 0.617075 | Time: 35.3521sec | Token/sec: 1853.810
    Step:   398 | Loss: 5.8250 | lr = 0.000384 | Norm = 0.425783 | Time: 35.2450sec | Token/sec: 1859.441
    Step:   399 | Loss: 5.8407 | lr = 0.000383 | Norm = 0.499143 | Time: 35.2012sec | Token/sec: 1861.753
    Validation loss: 5.7516
    Rank: 1 | Sample1: I am crazy man, and some examples, it makes me to the best thing that the reader to have never.
    And this is a story about a lot of these things to hear them as we
    Rank: 1 | Sample2: I am crazy man, but in a whole. So about and she could be doing a nice thing. A very old, no good is more.<|endoftext|>The only way in other languages of this word
    Rank: 1 | Sample3: I am crazy man, but all some of the day they made the day of that the time of his life I'm the rest of the first. I don't know of my. This seems to
    Rank: 0 | Sample1: I am crazy man, from the same year of the last year.
    After he wants at the last week it was so of the second time that I have to tell the house, I'm no
    Rank: 0 | Sample2: I am crazy man, it would have already seen in the world would give him to. At the whole time, a man who had the same bit, but of the same thing of those who.
    Rank: 0 | Sample3: I am crazy man, it was no doubt a short, in a half a half, a one would that, and was a man is not been placed about a well. The only a way,
    HellaSwag accuracy: 2383/10042=0.2373
    Step:   400 | Loss: 5.8781 | lr = 0.000382 | Norm = 0.486330 | Time: 367.1438sec | Token/sec: 178.502
    Step:   401 | Loss: 5.8002 | lr = 0.000381 | Norm = 0.474959 | Time: 34.9625sec | Token/sec: 1874.467
    Step:   402 | Loss: 5.7944 | lr = 0.000380 | Norm = 0.468496 | Time: 35.5020sec | Token/sec: 1845.980
    Step:   403 | Loss: 5.8529 | lr = 0.000379 | Norm = 0.422816 | Time: 34.9544sec | Token/sec: 1874.898
    Step:   404 | Loss: 5.7428 | lr = 0.000378 | Norm = 0.496369 | Time: 35.5135sec | Token/sec: 1845.381
    Step:   405 | Loss: 5.7956 | lr = 0.000377 | Norm = 0.388202 | Time: 35.2849sec | Token/sec: 1857.340
    Step:   406 | Loss: 5.7785 | lr = 0.000376 | Norm = 0.383500 | Time: 34.9273sec | Token/sec: 1876.355
    Step:   407 | Loss: 5.7643 | lr = 0.000376 | Norm = 0.419331 | Time: 35.2937sec | Token/sec: 1856.875
    Step:   408 | Loss: 5.7534 | lr = 0.000375 | Norm = 0.325967 | Time: 35.4912sec | Token/sec: 1846.544
    Step:   409 | Loss: 5.7132 | lr = 0.000374 | Norm = 0.423524 | Time: 35.4444sec | Token/sec: 1848.981
    Step:   410 | Loss: 5.7862 | lr = 0.000373 | Norm = 0.600415 | Time: 35.4727sec | Token/sec: 1847.505
    Step:   411 | Loss: 5.8087 | lr = 0.000372 | Norm = 0.454549 | Time: 35.4139sec | Token/sec: 1850.573
    Step:   412 | Loss: 5.7952 | lr = 0.000371 | Norm = 0.489547 | Time: 35.1957sec | Token/sec: 1862.046
    Step:   413 | Loss: 5.4437 | lr = 0.000370 | Norm = 0.765439 | Time: 35.1418sec | Token/sec: 1864.902
    Step:   414 | Loss: 5.7593 | lr = 0.000369 | Norm = 0.717032 | Time: 35.1141sec | Token/sec: 1866.371
    Step:   415 | Loss: 5.7026 | lr = 0.000368 | Norm = 0.408212 | Time: 35.1114sec | Token/sec: 1866.514
    Step:   416 | Loss: 5.8528 | lr = 0.000367 | Norm = 0.654422 | Time: 35.2366sec | Token/sec: 1859.887
    Step:   417 | Loss: 5.7286 | lr = 0.000366 | Norm = 0.487331 | Time: 35.4386sec | Token/sec: 1849.282
    Step:   418 | Loss: 5.8056 | lr = 0.000365 | Norm = 0.454268 | Time: 35.5061sec | Token/sec: 1845.766
    Step:   419 | Loss: 5.7303 | lr = 0.000364 | Norm = 0.507927 | Time: 35.4041sec | Token/sec: 1851.086
    Step:   420 | Loss: 5.6925 | lr = 0.000363 | Norm = 0.353515 | Time: 35.2738sec | Token/sec: 1857.925
    Step:   421 | Loss: 5.8052 | lr = 0.000362 | Norm = 0.456519 | Time: 35.2403sec | Token/sec: 1859.689
    Step:   422 | Loss: 5.7232 | lr = 0.000361 | Norm = 0.463768 | Time: 35.1669sec | Token/sec: 1863.572
    Step:   423 | Loss: 5.6604 | lr = 0.000360 | Norm = 0.383248 | Time: 35.1285sec | Token/sec: 1865.610
    Step:   424 | Loss: 5.7590 | lr = 0.000359 | Norm = 0.445837 | Time: 35.0980sec | Token/sec: 1867.229
    Step:   425 | Loss: 5.7726 | lr = 0.000359 | Norm = 0.404698 | Time: 35.0495sec | Token/sec: 1869.813
    Step:   426 | Loss: 5.6383 | lr = 0.000358 | Norm = 0.420516 | Time: 35.0176sec | Token/sec: 1871.517
    Step:   427 | Loss: 5.7187 | lr = 0.000357 | Norm = 0.513333 | Time: 35.2013sec | Token/sec: 1861.748
    Step:   428 | Loss: 5.6484 | lr = 0.000356 | Norm = 0.380552 | Time: 35.4048sec | Token/sec: 1851.049
    Step:   429 | Loss: 5.6369 | lr = 0.000355 | Norm = 0.410611 | Time: 35.4728sec | Token/sec: 1847.502
    Step:   430 | Loss: 5.7132 | lr = 0.000354 | Norm = 0.430030 | Time: 35.4760sec | Token/sec: 1847.331
    Step:   431 | Loss: 5.6852 | lr = 0.000353 | Norm = 0.383369 | Time: 35.4945sec | Token/sec: 1846.370
    Step:   432 | Loss: 5.7660 | lr = 0.000352 | Norm = 0.373329 | Time: 35.4050sec | Token/sec: 1851.037
    Step:   433 | Loss: 5.7070 | lr = 0.000351 | Norm = 0.322337 | Time: 35.2730sec | Token/sec: 1857.967
    Step:   434 | Loss: 5.7377 | lr = 0.000350 | Norm = 0.354073 | Time: 35.1711sec | Token/sec: 1863.346
    Step:   435 | Loss: 5.6174 | lr = 0.000349 | Norm = 0.385741 | Time: 35.0967sec | Token/sec: 1867.298
    Step:   436 | Loss: 5.7943 | lr = 0.000348 | Norm = 0.419757 | Time: 35.1357sec | Token/sec: 1865.224
    Step:   437 | Loss: 5.6555 | lr = 0.000347 | Norm = 0.398168 | Time: 35.2727sec | Token/sec: 1857.983
    Step:   438 | Loss: 5.6920 | lr = 0.000346 | Norm = 0.452166 | Time: 35.4159sec | Token/sec: 1850.469
    Step:   439 | Loss: 5.6211 | lr = 0.000345 | Norm = 0.459919 | Time: 35.4644sec | Token/sec: 1847.939
    Step:   440 | Loss: 5.5854 | lr = 0.000344 | Norm = 0.519102 | Time: 35.3760sec | Token/sec: 1852.555
    Step:   441 | Loss: 5.6889 | lr = 0.000343 | Norm = 0.417480 | Time: 35.5033sec | Token/sec: 1845.915
    Step:   442 | Loss: 5.7157 | lr = 0.000342 | Norm = 0.401685 | Time: 35.3597sec | Token/sec: 1853.410
    Step:   443 | Loss: 5.7098 | lr = 0.000341 | Norm = 0.370996 | Time: 35.2544sec | Token/sec: 1858.945
    Step:   444 | Loss: 5.6846 | lr = 0.000340 | Norm = 0.384004 | Time: 35.2280sec | Token/sec: 1860.337
    Step:   445 | Loss: 5.6368 | lr = 0.000340 | Norm = 0.393204 | Time: 35.1439sec | Token/sec: 1864.793
    Step:   446 | Loss: 5.6010 | lr = 0.000339 | Norm = 0.441173 | Time: 35.3107sec | Token/sec: 1855.983
    Step:   447 | Loss: 5.6643 | lr = 0.000338 | Norm = 0.389500 | Time: 35.4191sec | Token/sec: 1850.299
    Step:   448 | Loss: 5.6029 | lr = 0.000337 | Norm = 0.414697 | Time: 35.4481sec | Token/sec: 1848.786
    Step:   449 | Loss: 5.6356 | lr = 0.000336 | Norm = 0.343015 | Time: 35.4899sec | Token/sec: 1846.612
    Step:   450 | Loss: 5.6402 | lr = 0.000335 | Norm = 0.405642 | Time: 35.3888sec | Token/sec: 1851.887
    Step:   451 | Loss: 5.6330 | lr = 0.000334 | Norm = 0.376603 | Time: 35.1526sec | Token/sec: 1864.330
    Step:   452 | Loss: 5.6037 | lr = 0.000333 | Norm = 0.404038 | Time: 35.0383sec | Token/sec: 1870.409
    Step:   453 | Loss: 5.6124 | lr = 0.000332 | Norm = 0.418981 | Time: 35.0437sec | Token/sec: 1870.121
    Step:   454 | Loss: 5.6591 | lr = 0.000331 | Norm = 0.428613 | Time: 35.2167sec | Token/sec: 1860.935
    Step:   455 | Loss: 5.6289 | lr = 0.000330 | Norm = 0.362974 | Time: 35.3450sec | Token/sec: 1854.180
    Step:   456 | Loss: 5.6924 | lr = 0.000329 | Norm = 0.443189 | Time: 35.4512sec | Token/sec: 1848.626
    Step:   457 | Loss: 5.7244 | lr = 0.000328 | Norm = 0.514803 | Time: 35.4838sec | Token/sec: 1846.925
    Step:   458 | Loss: 5.7154 | lr = 0.000327 | Norm = 0.538445 | Time: 35.4636sec | Token/sec: 1847.977
    Step:   459 | Loss: 5.6358 | lr = 0.000326 | Norm = 0.464533 | Time: 35.4102sec | Token/sec: 1850.765
    Step:   460 | Loss: 5.6291 | lr = 0.000325 | Norm = 0.472631 | Time: 35.2843sec | Token/sec: 1857.372
    Step:   461 | Loss: 5.6249 | lr = 0.000324 | Norm = 0.545881 | Time: 35.2482sec | Token/sec: 1859.274
    Step:   462 | Loss: 5.7316 | lr = 0.000323 | Norm = 0.397759 | Time: 35.1611sec | Token/sec: 1863.877
    Step:   463 | Loss: 5.6770 | lr = 0.000322 | Norm = 0.474255 | Time: 35.1120sec | Token/sec: 1866.487
    Step:   464 | Loss: 5.6375 | lr = 0.000321 | Norm = 0.376863 | Time: 35.1016sec | Token/sec: 1867.035
    Step:   465 | Loss: 5.5838 | lr = 0.000320 | Norm = 0.428202 | Time: 35.0892sec | Token/sec: 1867.697
    Step:   466 | Loss: 5.7120 | lr = 0.000320 | Norm = 0.687863 | Time: 35.0558sec | Token/sec: 1869.478
    Step:   467 | Loss: 5.5611 | lr = 0.000319 | Norm = 0.400607 | Time: 35.0757sec | Token/sec: 1868.418
    Step:   468 | Loss: 5.5479 | lr = 0.000318 | Norm = 0.471366 | Time: 35.0793sec | Token/sec: 1868.226
    Step:   469 | Loss: 5.5418 | lr = 0.000317 | Norm = 0.411961 | Time: 35.0804sec | Token/sec: 1868.165
    Step:   470 | Loss: 5.6260 | lr = 0.000316 | Norm = 0.515550 | Time: 35.1539sec | Token/sec: 1864.257
    Step:   471 | Loss: 5.5344 | lr = 0.000315 | Norm = 0.369337 | Time: 35.3131sec | Token/sec: 1855.856
    Step:   472 | Loss: 5.6269 | lr = 0.000314 | Norm = 0.483962 | Time: 35.4293sec | Token/sec: 1849.770
    Step:   473 | Loss: 5.5769 | lr = 0.000313 | Norm = 0.457840 | Time: 35.4650sec | Token/sec: 1847.905
    Step:   474 | Loss: 5.5131 | lr = 0.000312 | Norm = 0.376126 | Time: 35.5315sec | Token/sec: 1844.446
    Step:   475 | Loss: 5.5378 | lr = 0.000311 | Norm = 0.526788 | Time: 35.4680sec | Token/sec: 1847.750
    Step:   476 | Loss: 5.5417 | lr = 0.000310 | Norm = 0.452490 | Time: 35.2872sec | Token/sec: 1857.217
    Step:   477 | Loss: 5.6204 | lr = 0.000309 | Norm = 0.424713 | Time: 35.1527sec | Token/sec: 1864.322
    Step:   478 | Loss: 5.5298 | lr = 0.000308 | Norm = 0.381289 | Time: 35.1138sec | Token/sec: 1866.390
    Step:   479 | Loss: 5.6040 | lr = 0.000307 | Norm = 0.380449 | Time: 35.0388sec | Token/sec: 1870.386
    Step:   480 | Loss: 5.4826 | lr = 0.000306 | Norm = 0.400490 | Time: 35.0567sec | Token/sec: 1869.430
    Step:   481 | Loss: 5.5818 | lr = 0.000305 | Norm = 0.423818 | Time: 35.1706sec | Token/sec: 1863.376
    Step:   482 | Loss: 5.5331 | lr = 0.000304 | Norm = 0.406011 | Time: 35.3558sec | Token/sec: 1853.611
    Step:   483 | Loss: 5.6415 | lr = 0.000303 | Norm = 0.363060 | Time: 35.4226sec | Token/sec: 1850.121
    Step:   484 | Loss: 5.4545 | lr = 0.000302 | Norm = 0.440152 | Time: 35.4597sec | Token/sec: 1848.182
    Step:   485 | Loss: 5.4269 | lr = 0.000301 | Norm = 0.421236 | Time: 35.4941sec | Token/sec: 1846.391
    Step:   486 | Loss: 5.5983 | lr = 0.000301 | Norm = 0.430349 | Time: 35.4980sec | Token/sec: 1846.189
    Step:   487 | Loss: 5.5026 | lr = 0.000300 | Norm = 0.420030 | Time: 35.3457sec | Token/sec: 1854.146
    Step:   488 | Loss: 5.4455 | lr = 0.000299 | Norm = 0.424835 | Time: 35.2357sec | Token/sec: 1859.929
    Step:   489 | Loss: 5.4814 | lr = 0.000298 | Norm = 0.469543 | Time: 35.1623sec | Token/sec: 1863.815
    Step:   490 | Loss: 5.5864 | lr = 0.000297 | Norm = 0.415720 | Time: 35.1131sec | Token/sec: 1866.426
    Step:   491 | Loss: 5.5917 | lr = 0.000296 | Norm = 0.417432 | Time: 35.1566sec | Token/sec: 1864.115
    Step:   492 | Loss: 5.6105 | lr = 0.000295 | Norm = 0.606148 | Time: 35.3454sec | Token/sec: 1854.157
    Step:   493 | Loss: 5.5405 | lr = 0.000294 | Norm = 0.453295 | Time: 35.4512sec | Token/sec: 1848.624
    Step:   494 | Loss: 5.5259 | lr = 0.000293 | Norm = 0.441583 | Time: 35.4672sec | Token/sec: 1847.794
    Step:   495 | Loss: 5.4217 | lr = 0.000292 | Norm = 0.400862 | Time: 35.5309sec | Token/sec: 1844.478
    Step:   496 | Loss: 5.5194 | lr = 0.000291 | Norm = 0.371460 | Time: 35.4452sec | Token/sec: 1848.938
    Step:   497 | Loss: 5.4479 | lr = 0.000290 | Norm = 0.433005 | Time: 35.3170sec | Token/sec: 1855.649
    Step:   498 | Loss: 5.5862 | lr = 0.000289 | Norm = 0.387455 | Time: 35.1137sec | Token/sec: 1866.396
    Step:   499 | Loss: 5.5666 | lr = 0.000288 | Norm = 0.400580 | Time: 35.0780sec | Token/sec: 1868.295
    Validation loss: 5.5520
    Rank: 1 | Sample1: I am crazy man, so well put a lot of children in some more importantly and more money.
    To get one day, you, you can be a very little to talk in other people who
    Rank: 1 | Sample2: I am crazy man, I would not do, I can't be able to look to use others to know more so. But I feel that they are not even you think about it can not change
    Rank: 1 | Sample3: I am crazy man, I saw his father that the way of him, he was going on to try them. Then, she told when I don't get it to do with him, with you
    Rank: 0 | Sample1: I am crazy man, is said in a dog that are also referred not able to understand the other words, and not only for them.
    In fact, the two different number of years old that
    Rank: 0 | Sample2: I am crazy man, and how he was said.
    What you live? Please know? Or that day, you know about a teacher will be a bit more than kids for someone.
    These
    Rank: 0 | Sample3: I am crazy man, and maybe it is a person?
    - You also make an idea, but that if you can try the message that this’re going from it. As you should
    HellaSwag accuracy: 2357/10042=0.2347
    Step:   500 | Loss: 5.4526 | lr = 0.000287 | Norm = 0.423024 | Time: 367.1652sec | Token/sec: 178.492
    Step:   501 | Loss: 5.5718 | lr = 0.000286 | Norm = 0.374797 | Time: 35.1652sec | Token/sec: 1863.659
    Step:   502 | Loss: 5.5673 | lr = 0.000285 | Norm = 0.503150 | Time: 35.1636sec | Token/sec: 1863.744
    Step:   503 | Loss: 5.5138 | lr = 0.000284 | Norm = 0.426454 | Time: 34.8880sec | Token/sec: 1878.471
    Step:   504 | Loss: 5.5428 | lr = 0.000284 | Norm = 0.476494 | Time: 35.3161sec | Token/sec: 1855.695
    Step:   505 | Loss: 5.4718 | lr = 0.000283 | Norm = 0.495257 | Time: 35.5164sec | Token/sec: 1845.230
    Step:   506 | Loss: 5.5389 | lr = 0.000282 | Norm = 0.410931 | Time: 35.4840sec | Token/sec: 1846.915
    Step:   507 | Loss: 5.4956 | lr = 0.000281 | Norm = 0.478405 | Time: 35.0983sec | Token/sec: 1867.211
    Step:   508 | Loss: 5.4215 | lr = 0.000280 | Norm = 0.410257 | Time: 35.0456sec | Token/sec: 1870.020
    Step:   509 | Loss: 5.5124 | lr = 0.000279 | Norm = 0.400243 | Time: 34.9701sec | Token/sec: 1874.058
    Step:   510 | Loss: 5.4560 | lr = 0.000278 | Norm = 0.402460 | Time: 35.0655sec | Token/sec: 1868.961
    Step:   511 | Loss: 5.5241 | lr = 0.000277 | Norm = 0.399694 | Time: 35.3404sec | Token/sec: 1854.423
    Step:   512 | Loss: 5.4370 | lr = 0.000276 | Norm = 0.457485 | Time: 35.4553sec | Token/sec: 1848.412
    Step:   513 | Loss: 5.4190 | lr = 0.000275 | Norm = 0.368804 | Time: 35.4676sec | Token/sec: 1847.769
    Step:   514 | Loss: 5.4157 | lr = 0.000274 | Norm = 0.363841 | Time: 35.4770sec | Token/sec: 1847.281
    Step:   515 | Loss: 5.3600 | lr = 0.000273 | Norm = 0.387793 | Time: 35.2511sec | Token/sec: 1859.121
    Step:   516 | Loss: 5.3768 | lr = 0.000272 | Norm = 0.364599 | Time: 35.0817sec | Token/sec: 1868.094
    Step:   517 | Loss: 5.3705 | lr = 0.000271 | Norm = 0.422025 | Time: 35.0512sec | Token/sec: 1869.724
    Step:   518 | Loss: 5.4355 | lr = 0.000270 | Norm = 0.396814 | Time: 34.9728sec | Token/sec: 1873.912
    Step:   519 | Loss: 5.4046 | lr = 0.000270 | Norm = 0.454337 | Time: 35.0973sec | Token/sec: 1867.265
    Step:   520 | Loss: 5.4502 | lr = 0.000269 | Norm = 0.405726 | Time: 35.4073sec | Token/sec: 1850.920
    Step:   521 | Loss: 5.5388 | lr = 0.000268 | Norm = 0.420794 | Time: 35.4932sec | Token/sec: 1846.438
    Step:   522 | Loss: 5.3796 | lr = 0.000267 | Norm = 0.399016 | Time: 35.5048sec | Token/sec: 1845.837
    Step:   523 | Loss: 5.4479 | lr = 0.000266 | Norm = 0.436855 | Time: 35.2695sec | Token/sec: 1858.149
    Step:   524 | Loss: 5.3469 | lr = 0.000265 | Norm = 0.429776 | Time: 34.9908sec | Token/sec: 1872.950
    Step:   525 | Loss: 5.4228 | lr = 0.000264 | Norm = 0.451333 | Time: 35.0166sec | Token/sec: 1871.567
    Step:   526 | Loss: 5.4368 | lr = 0.000263 | Norm = 0.443436 | Time: 35.2367sec | Token/sec: 1859.878
    Step:   527 | Loss: 5.4428 | lr = 0.000262 | Norm = 0.402430 | Time: 35.3648sec | Token/sec: 1853.141
    Step:   528 | Loss: 5.3599 | lr = 0.000261 | Norm = 0.414764 | Time: 35.4597sec | Token/sec: 1848.184
    Step:   529 | Loss: 5.3990 | lr = 0.000260 | Norm = 0.425154 | Time: 35.4632sec | Token/sec: 1847.999
    Step:   530 | Loss: 5.4608 | lr = 0.000259 | Norm = 0.359339 | Time: 35.4371sec | Token/sec: 1849.364
    Step:   531 | Loss: 5.5980 | lr = 0.000258 | Norm = 0.498267 | Time: 35.3761sec | Token/sec: 1852.550
    Step:   532 | Loss: 5.3092 | lr = 0.000258 | Norm = 0.576971 | Time: 35.2252sec | Token/sec: 1860.485
    Step:   533 | Loss: 5.4004 | lr = 0.000257 | Norm = 0.530559 | Time: 35.1832sec | Token/sec: 1862.708
    Step:   534 | Loss: 5.3238 | lr = 0.000256 | Norm = 0.431529 | Time: 35.1079sec | Token/sec: 1866.704
    Step:   535 | Loss: 5.4620 | lr = 0.000255 | Norm = 0.496303 | Time: 35.1255sec | Token/sec: 1865.769
    Step:   536 | Loss: 5.3732 | lr = 0.000254 | Norm = 0.522970 | Time: 35.2126sec | Token/sec: 1861.154
    Step:   537 | Loss: 5.3629 | lr = 0.000253 | Norm = 0.434434 | Time: 35.4072sec | Token/sec: 1850.923
    Step:   538 | Loss: 5.4550 | lr = 0.000252 | Norm = 0.739906 | Time: 35.4330sec | Token/sec: 1849.573
    Step:   539 | Loss: 5.4017 | lr = 0.000251 | Norm = 0.737219 | Time: 35.4229sec | Token/sec: 1850.103
    Step:   540 | Loss: 5.4358 | lr = 0.000250 | Norm = 0.535388 | Time: 35.4932sec | Token/sec: 1846.437
    Step:   541 | Loss: 5.4213 | lr = 0.000249 | Norm = 0.542521 | Time: 35.3893sec | Token/sec: 1851.857
    Step:   542 | Loss: 5.3884 | lr = 0.000248 | Norm = 0.531650 | Time: 35.2353sec | Token/sec: 1859.955
    Step:   543 | Loss: 5.4301 | lr = 0.000247 | Norm = 0.471292 | Time: 35.1342sec | Token/sec: 1865.307
    Step:   544 | Loss: 5.3704 | lr = 0.000247 | Norm = 0.490639 | Time: 35.2129sec | Token/sec: 1861.136
    Step:   545 | Loss: 5.3985 | lr = 0.000246 | Norm = 0.424312 | Time: 35.4174sec | Token/sec: 1850.391
    Step:   546 | Loss: 5.2956 | lr = 0.000245 | Norm = 0.398890 | Time: 35.4632sec | Token/sec: 1848.003
    Step:   547 | Loss: 5.4588 | lr = 0.000244 | Norm = 0.441778 | Time: 35.4747sec | Token/sec: 1847.402
    Step:   548 | Loss: 5.4243 | lr = 0.000243 | Norm = 0.431221 | Time: 35.4664sec | Token/sec: 1847.833
    Step:   549 | Loss: 5.3525 | lr = 0.000242 | Norm = 0.402633 | Time: 35.4566sec | Token/sec: 1848.344
    Step:   550 | Loss: 5.3879 | lr = 0.000241 | Norm = 0.487957 | Time: 35.1997sec | Token/sec: 1861.832
    Step:   551 | Loss: 5.4637 | lr = 0.000240 | Norm = 0.481469 | Time: 35.1503sec | Token/sec: 1864.453
    Step:   552 | Loss: 5.3013 | lr = 0.000239 | Norm = 0.471934 | Time: 35.0787sec | Token/sec: 1868.257
    Step:   553 | Loss: 5.4609 | lr = 0.000238 | Norm = 0.423158 | Time: 35.1907sec | Token/sec: 1862.310
    Step:   554 | Loss: 5.3359 | lr = 0.000238 | Norm = 0.429024 | Time: 35.4630sec | Token/sec: 1848.012
    Step:   555 | Loss: 5.3852 | lr = 0.000237 | Norm = 0.349508 | Time: 35.4516sec | Token/sec: 1848.605
    Step:   556 | Loss: 5.3114 | lr = 0.000236 | Norm = 0.378508 | Time: 35.4951sec | Token/sec: 1846.340
    Step:   557 | Loss: 5.3937 | lr = 0.000235 | Norm = 0.365819 | Time: 35.4673sec | Token/sec: 1847.789
    Step:   558 | Loss: 5.5615 | lr = 0.000234 | Norm = 0.402097 | Time: 35.4114sec | Token/sec: 1850.706
    Step:   559 | Loss: 5.6505 | lr = 0.000233 | Norm = 0.578165 | Time: 35.2155sec | Token/sec: 1861.001
    Step:   560 | Loss: 5.5480 | lr = 0.000232 | Norm = 0.493255 | Time: 35.1335sec | Token/sec: 1865.340
    Step:   561 | Loss: 5.6649 | lr = 0.000231 | Norm = 0.463440 | Time: 35.1803sec | Token/sec: 1862.860
    Step:   562 | Loss: 5.5385 | lr = 0.000230 | Norm = 0.425795 | Time: 35.0822sec | Token/sec: 1868.071
    Step:   563 | Loss: 5.6415 | lr = 0.000230 | Norm = 0.532341 | Time: 35.3472sec | Token/sec: 1854.065
    Step:   564 | Loss: 5.4612 | lr = 0.000229 | Norm = 0.478735 | Time: 35.4434sec | Token/sec: 1849.033
    Step:   565 | Loss: 5.4218 | lr = 0.000228 | Norm = 0.543696 | Time: 35.4931sec | Token/sec: 1846.446
    Step:   566 | Loss: 5.5654 | lr = 0.000227 | Norm = 0.573045 | Time: 35.4684sec | Token/sec: 1847.730
    Step:   567 | Loss: 5.5682 | lr = 0.000226 | Norm = 0.411240 | Time: 35.5337sec | Token/sec: 1844.335
    Step:   568 | Loss: 5.6144 | lr = 0.000225 | Norm = 0.448597 | Time: 35.2968sec | Token/sec: 1856.710
    Step:   569 | Loss: 5.5555 | lr = 0.000224 | Norm = 0.413953 | Time: 35.2521sec | Token/sec: 1859.069
    Step:   570 | Loss: 5.4964 | lr = 0.000223 | Norm = 0.420287 | Time: 35.1016sec | Token/sec: 1867.038
    Step:   571 | Loss: 5.4585 | lr = 0.000223 | Norm = 0.383583 | Time: 35.1259sec | Token/sec: 1865.747
    Step:   572 | Loss: 5.4403 | lr = 0.000222 | Norm = 0.411712 | Time: 35.0679sec | Token/sec: 1868.832
    Step:   573 | Loss: 5.5025 | lr = 0.000221 | Norm = 0.373825 | Time: 35.0037sec | Token/sec: 1872.258
    Step:   574 | Loss: 5.5095 | lr = 0.000220 | Norm = 0.397867 | Time: 35.2352sec | Token/sec: 1859.960
    Step:   575 | Loss: 5.4549 | lr = 0.000219 | Norm = 0.397058 | Time: 35.4563sec | Token/sec: 1848.358
    Step:   576 | Loss: 5.4589 | lr = 0.000218 | Norm = 0.345131 | Time: 35.4673sec | Token/sec: 1847.787
    Step:   577 | Loss: 5.5056 | lr = 0.000217 | Norm = 0.442697 | Time: 35.4956sec | Token/sec: 1846.315
    Step:   578 | Loss: 5.4672 | lr = 0.000216 | Norm = 0.443704 | Time: 35.4105sec | Token/sec: 1850.751
    Step:   579 | Loss: 5.4704 | lr = 0.000216 | Norm = 0.414327 | Time: 35.3263sec | Token/sec: 1855.162
    Step:   580 | Loss: 5.4917 | lr = 0.000215 | Norm = 0.523523 | Time: 35.2126sec | Token/sec: 1861.151
    Step:   581 | Loss: 5.4597 | lr = 0.000214 | Norm = 0.413924 | Time: 35.1917sec | Token/sec: 1862.260
    Step:   582 | Loss: 5.4366 | lr = 0.000213 | Norm = 0.451244 | Time: 35.1544sec | Token/sec: 1864.234
    Step:   583 | Loss: 5.3842 | lr = 0.000212 | Norm = 0.381074 | Time: 35.1507sec | Token/sec: 1864.429
    Step:   584 | Loss: 5.5023 | lr = 0.000211 | Norm = 0.485374 | Time: 35.1117sec | Token/sec: 1866.500
    Step:   585 | Loss: 5.4374 | lr = 0.000210 | Norm = 0.399613 | Time: 35.1599sec | Token/sec: 1863.941
    Step:   586 | Loss: 5.4641 | lr = 0.000210 | Norm = 0.381602 | Time: 35.0273sec | Token/sec: 1870.999
    Step:   587 | Loss: 5.4869 | lr = 0.000209 | Norm = 0.433440 | Time: 35.0977sec | Token/sec: 1867.244
    Step:   588 | Loss: 5.5192 | lr = 0.000208 | Norm = 0.342673 | Time: 35.1100sec | Token/sec: 1866.589
    Step:   589 | Loss: 5.4940 | lr = 0.000207 | Norm = 0.423415 | Time: 35.2822sec | Token/sec: 1857.479
    Step:   590 | Loss: 5.4984 | lr = 0.000206 | Norm = 0.387509 | Time: 35.4013sec | Token/sec: 1851.233
    Step:   591 | Loss: 5.4204 | lr = 0.000205 | Norm = 0.373106 | Time: 35.4525sec | Token/sec: 1848.557
    Step:   592 | Loss: 5.4742 | lr = 0.000204 | Norm = 0.395338 | Time: 35.4218sec | Token/sec: 1850.158
    Step:   593 | Loss: 5.4628 | lr = 0.000204 | Norm = 0.350815 | Time: 35.5291sec | Token/sec: 1844.574
    Step:   594 | Loss: 5.6946 | lr = 0.000203 | Norm = 0.498747 | Time: 35.4623sec | Token/sec: 1848.046
    Step:   595 | Loss: 5.3937 | lr = 0.000202 | Norm = 0.486707 | Time: 35.2621sec | Token/sec: 1858.539
    Step:   596 | Loss: 5.4479 | lr = 0.000201 | Norm = 0.493469 | Time: 35.0831sec | Token/sec: 1868.024
    Step:   597 | Loss: 5.4564 | lr = 0.000200 | Norm = 0.423274 | Time: 35.1515sec | Token/sec: 1864.388
    Step:   598 | Loss: 5.4896 | lr = 0.000199 | Norm = 0.533067 | Time: 35.0159sec | Token/sec: 1871.608
    Step:   599 | Loss: 5.4405 | lr = 0.000199 | Norm = 0.596271 | Time: 35.3537sec | Token/sec: 1853.726
    Validation loss: 5.3944
    Rank: 1 | Sample1: I am crazy man, for such that the same time it is to be made more comfortable, so this time is, not to get it by the second day.
    For those old, you can
    Rank: 1 | Sample2: I am crazy man,
    "This is known that there is one country; but, and we are so, we always
    with a
    all, for all and the most of the men of
    Rank: 1 | Sample3: I am crazy man, and he could get no idea you in their way, “they will not do’t leave them as you would feel at a friend you.” (18
    Rank: 0 | Sample1: I am crazy man, as a single woman, so, though the children had long seen and the new man on all of them. It was a few sons who were living. And she was so
    Rank: 0 | Sample2: I am crazy man, she would have ever been too well from my and the world's great woman. It’s easy, according to its “to live, and they make us of
    Rank: 0 | Sample3: I am crazy man, a small-life as part of the British. It has been made to be so a large enough than people, the best for that the country than that were not in a
    HellaSwag accuracy: 2342/10042=0.2332
    Step:   600 | Loss: 5.4219 | lr = 0.000198 | Norm = 0.451644 | Time: 367.7159sec | Token/sec: 178.225
    Step:   601 | Loss: 5.4964 | lr = 0.000197 | Norm = 0.464785 | Time: 35.1494sec | Token/sec: 1864.499
    Step:   602 | Loss: 5.4883 | lr = 0.000196 | Norm = 0.425498 | Time: 35.2668sec | Token/sec: 1858.290
    Step:   603 | Loss: 5.4206 | lr = 0.000195 | Norm = 0.430353 | Time: 35.3336sec | Token/sec: 1854.780
    Step:   604 | Loss: 5.3375 | lr = 0.000194 | Norm = 0.435867 | Time: 35.4540sec | Token/sec: 1848.478
    Step:   605 | Loss: 5.3414 | lr = 0.000194 | Norm = 0.360510 | Time: 35.4507sec | Token/sec: 1848.651
    Step:   606 | Loss: 5.4045 | lr = 0.000193 | Norm = 0.408942 | Time: 35.4443sec | Token/sec: 1848.986
    Step:   607 | Loss: 5.3807 | lr = 0.000192 | Norm = 0.357556 | Time: 35.4692sec | Token/sec: 1847.690
    Step:   608 | Loss: 5.3247 | lr = 0.000191 | Norm = 0.334321 | Time: 35.4638sec | Token/sec: 1847.967
    Step:   609 | Loss: 5.3701 | lr = 0.000190 | Norm = 0.363121 | Time: 35.4875sec | Token/sec: 1846.737
    Step:   610 | Loss: 5.5409 | lr = 0.000190 | Norm = 0.394149 | Time: 35.4468sec | Token/sec: 1848.855
    Step:   611 | Loss: 5.4093 | lr = 0.000189 | Norm = 0.411988 | Time: 35.4308sec | Token/sec: 1849.691
    Step:   612 | Loss: 5.4132 | lr = 0.000188 | Norm = 0.356122 | Time: 35.4488sec | Token/sec: 1848.752
    Step:   613 | Loss: 5.4911 | lr = 0.000187 | Norm = 0.484240 | Time: 35.4057sec | Token/sec: 1851.000
    Step:   614 | Loss: 5.4423 | lr = 0.000186 | Norm = 0.405646 | Time: 35.4578sec | Token/sec: 1848.281
    Step:   615 | Loss: 5.4537 | lr = 0.000185 | Norm = 0.417881 | Time: 35.4570sec | Token/sec: 1848.324
    Step:   616 | Loss: 5.4290 | lr = 0.000185 | Norm = 0.380538 | Time: 35.4575sec | Token/sec: 1848.298
    Step:   617 | Loss: 5.2824 | lr = 0.000184 | Norm = 0.414126 | Time: 35.4583sec | Token/sec: 1848.257
    Step:   618 | Loss: 5.3690 | lr = 0.000183 | Norm = 0.419153 | Time: 35.3978sec | Token/sec: 1851.413
    Step:   619 | Loss: 5.3719 | lr = 0.000182 | Norm = 0.396498 | Time: 35.3486sec | Token/sec: 1853.991
    Step:   620 | Loss: 5.3796 | lr = 0.000181 | Norm = 0.354180 | Time: 35.4281sec | Token/sec: 1849.831
    Step:   621 | Loss: 5.4140 | lr = 0.000181 | Norm = 0.436288 | Time: 35.3947sec | Token/sec: 1851.579
    Step:   622 | Loss: 5.3441 | lr = 0.000180 | Norm = 0.383405 | Time: 35.3708sec | Token/sec: 1852.827
    Step:   623 | Loss: 5.3178 | lr = 0.000179 | Norm = 0.427908 | Time: 35.4330sec | Token/sec: 1849.576
    Step:   624 | Loss: 5.3779 | lr = 0.000178 | Norm = 0.400279 | Time: 35.4533sec | Token/sec: 1848.515
    Step:   625 | Loss: 5.4451 | lr = 0.000178 | Norm = 0.468981 | Time: 35.4053sec | Token/sec: 1851.020
    Step:   626 | Loss: 5.3673 | lr = 0.000177 | Norm = 0.418900 | Time: 35.3431sec | Token/sec: 1854.279
    Step:   627 | Loss: 5.2957 | lr = 0.000176 | Norm = 0.564665 | Time: 35.4505sec | Token/sec: 1848.661
    Step:   628 | Loss: 5.4447 | lr = 0.000175 | Norm = 0.461168 | Time: 35.4067sec | Token/sec: 1850.948
    Step:   629 | Loss: 5.5005 | lr = 0.000174 | Norm = 0.431900 | Time: 35.4062sec | Token/sec: 1850.975
    Step:   630 | Loss: 5.4389 | lr = 0.000174 | Norm = 0.561270 | Time: 35.4659sec | Token/sec: 1847.862
    Step:   631 | Loss: 5.4234 | lr = 0.000173 | Norm = 0.419959 | Time: 35.4293sec | Token/sec: 1849.771
    Step:   632 | Loss: 5.6783 | lr = 0.000172 | Norm = 0.551739 | Time: 35.4654sec | Token/sec: 1847.884
    Step:   633 | Loss: 5.4405 | lr = 0.000171 | Norm = 0.602897 | Time: 35.4297sec | Token/sec: 1849.750
    Step:   634 | Loss: 5.3946 | lr = 0.000171 | Norm = 0.429374 | Time: 35.4190sec | Token/sec: 1850.305
    Step:   635 | Loss: 5.3872 | lr = 0.000170 | Norm = 0.604581 | Time: 35.4174sec | Token/sec: 1850.389
    Step:   636 | Loss: 5.4708 | lr = 0.000169 | Norm = 0.426139 | Time: 35.3818sec | Token/sec: 1852.253
    Step:   637 | Loss: 5.3394 | lr = 0.000168 | Norm = 0.475980 | Time: 35.4301sec | Token/sec: 1849.724
    Step:   638 | Loss: 5.3332 | lr = 0.000167 | Norm = 0.396926 | Time: 35.4513sec | Token/sec: 1848.621
    Step:   639 | Loss: 5.3998 | lr = 0.000167 | Norm = 0.415666 | Time: 35.4313sec | Token/sec: 1849.665
    Step:   640 | Loss: 5.3207 | lr = 0.000166 | Norm = 0.431417 | Time: 35.4569sec | Token/sec: 1848.330
    Step:   641 | Loss: 5.2914 | lr = 0.000165 | Norm = 0.465548 | Time: 35.4509sec | Token/sec: 1848.643
    Step:   642 | Loss: 5.2741 | lr = 0.000164 | Norm = 0.404344 | Time: 35.3750sec | Token/sec: 1852.608
    Step:   643 | Loss: 5.3740 | lr = 0.000164 | Norm = 0.381704 | Time: 35.4482sec | Token/sec: 1848.780
    Step:   644 | Loss: 5.4891 | lr = 0.000163 | Norm = 0.422276 | Time: 35.4421sec | Token/sec: 1849.098
    Step:   645 | Loss: 5.4049 | lr = 0.000162 | Norm = 0.388502 | Time: 35.3758sec | Token/sec: 1852.564
    Step:   646 | Loss: 5.3638 | lr = 0.000161 | Norm = 0.456625 | Time: 35.4071sec | Token/sec: 1850.927
    Step:   647 | Loss: 5.4106 | lr = 0.000161 | Norm = 0.395430 | Time: 35.3889sec | Token/sec: 1851.881
    Step:   648 | Loss: 5.3323 | lr = 0.000160 | Norm = 0.401144 | Time: 35.3934sec | Token/sec: 1851.647
    Step:   649 | Loss: 5.3741 | lr = 0.000159 | Norm = 0.396156 | Time: 35.3646sec | Token/sec: 1853.152
    Step:   650 | Loss: 5.2724 | lr = 0.000158 | Norm = 0.381508 | Time: 35.3564sec | Token/sec: 1853.584
    Step:   651 | Loss: 5.3253 | lr = 0.000158 | Norm = 0.393071 | Time: 35.3661sec | Token/sec: 1853.076
    Step:   652 | Loss: 5.2884 | lr = 0.000157 | Norm = 0.378962 | Time: 35.3925sec | Token/sec: 1851.690
    Step:   653 | Loss: 5.2673 | lr = 0.000156 | Norm = 0.370116 | Time: 35.3999sec | Token/sec: 1851.306
    Step:   654 | Loss: 5.2977 | lr = 0.000156 | Norm = 0.386859 | Time: 35.4185sec | Token/sec: 1850.334
    Step:   655 | Loss: 5.3267 | lr = 0.000155 | Norm = 0.383768 | Time: 35.4454sec | Token/sec: 1848.931
    Step:   656 | Loss: 5.1661 | lr = 0.000154 | Norm = 0.400781 | Time: 35.4637sec | Token/sec: 1847.972
    Step:   657 | Loss: 5.2534 | lr = 0.000153 | Norm = 0.378691 | Time: 35.4183sec | Token/sec: 1850.345
    Step:   658 | Loss: 5.3480 | lr = 0.000153 | Norm = 0.398253 | Time: 35.3799sec | Token/sec: 1852.349
    Step:   659 | Loss: 5.3383 | lr = 0.000152 | Norm = 0.402853 | Time: 35.4136sec | Token/sec: 1850.591
    Step:   660 | Loss: 5.3487 | lr = 0.000151 | Norm = 0.399394 | Time: 35.2718sec | Token/sec: 1858.028
    Step:   661 | Loss: 5.2691 | lr = 0.000151 | Norm = 0.360744 | Time: 35.3590sec | Token/sec: 1853.444
    Step:   662 | Loss: 5.3856 | lr = 0.000150 | Norm = 0.428006 | Time: 35.4096sec | Token/sec: 1850.798
    Step:   663 | Loss: 5.3165 | lr = 0.000149 | Norm = 0.377648 | Time: 35.3882sec | Token/sec: 1851.916
    Step:   664 | Loss: 5.2683 | lr = 0.000148 | Norm = 0.378354 | Time: 35.4448sec | Token/sec: 1848.958
    Step:   665 | Loss: 5.2966 | lr = 0.000148 | Norm = 0.362536 | Time: 35.2707sec | Token/sec: 1858.087
    Step:   666 | Loss: 5.1840 | lr = 0.000147 | Norm = 0.658200 | Time: 35.1155sec | Token/sec: 1866.301
    Step:   667 | Loss: 5.1517 | lr = 0.000146 | Norm = 0.486626 | Time: 34.9925sec | Token/sec: 1872.857
    Step:   668 | Loss: 5.3382 | lr = 0.000146 | Norm = 0.584865 | Time: 35.4854sec | Token/sec: 1846.846
    Step:   669 | Loss: 5.2449 | lr = 0.000145 | Norm = 0.393821 | Time: 35.3417sec | Token/sec: 1854.355
    Step:   670 | Loss: 5.3743 | lr = 0.000144 | Norm = 0.479546 | Time: 35.1608sec | Token/sec: 1863.895
    Step:   671 | Loss: 5.2543 | lr = 0.000144 | Norm = 0.466458 | Time: 35.0961sec | Token/sec: 1867.330
    Step:   672 | Loss: 5.3216 | lr = 0.000143 | Norm = 0.400504 | Time: 35.1540sec | Token/sec: 1864.254
    Step:   673 | Loss: 5.3453 | lr = 0.000142 | Norm = 0.430405 | Time: 35.4585sec | Token/sec: 1848.243
    Step:   674 | Loss: 5.3155 | lr = 0.000141 | Norm = 0.449164 | Time: 35.5877sec | Token/sec: 1841.537
    Step:   675 | Loss: 5.2232 | lr = 0.000141 | Norm = 0.478461 | Time: 35.4772sec | Token/sec: 1847.272
    Step:   676 | Loss: 5.2168 | lr = 0.000140 | Norm = 0.420743 | Time: 35.5004sec | Token/sec: 1846.062
    Step:   677 | Loss: 5.3707 | lr = 0.000139 | Norm = 0.525139 | Time: 35.5681sec | Token/sec: 1842.551
    Step:   678 | Loss: 5.2182 | lr = 0.000139 | Norm = 0.422422 | Time: 35.4368sec | Token/sec: 1849.377
    Step:   679 | Loss: 5.2453 | lr = 0.000138 | Norm = 0.423795 | Time: 35.4458sec | Token/sec: 1848.907
    Step:   680 | Loss: 5.2448 | lr = 0.000137 | Norm = 0.432185 | Time: 35.4746sec | Token/sec: 1847.408
    Step:   681 | Loss: 5.2691 | lr = 0.000137 | Norm = 0.349454 | Time: 35.4610sec | Token/sec: 1848.115
    Step:   682 | Loss: 5.2843 | lr = 0.000136 | Norm = 0.410412 | Time: 35.4045sec | Token/sec: 1851.066
    Step:   683 | Loss: 5.2353 | lr = 0.000135 | Norm = 0.420042 | Time: 35.4542sec | Token/sec: 1848.471
    Step:   684 | Loss: 5.2094 | lr = 0.000135 | Norm = 0.370525 | Time: 35.4077sec | Token/sec: 1850.898
    Step:   685 | Loss: 5.2873 | lr = 0.000134 | Norm = 0.353906 | Time: 35.3876sec | Token/sec: 1851.948
    Step:   686 | Loss: 5.3534 | lr = 0.000133 | Norm = 0.413801 | Time: 35.4433sec | Token/sec: 1849.038
    Step:   687 | Loss: 5.2997 | lr = 0.000133 | Norm = 0.406601 | Time: 35.4144sec | Token/sec: 1850.548
    Step:   688 | Loss: 5.2353 | lr = 0.000132 | Norm = 0.457571 | Time: 35.4525sec | Token/sec: 1848.558
    Step:   689 | Loss: 5.2478 | lr = 0.000131 | Norm = 0.448083 | Time: 35.4717sec | Token/sec: 1847.558
    Step:   690 | Loss: 5.1979 | lr = 0.000131 | Norm = 0.452585 | Time: 35.4476sec | Token/sec: 1848.812
    Step:   691 | Loss: 5.2801 | lr = 0.000130 | Norm = 0.382447 | Time: 35.4683sec | Token/sec: 1847.733
    Step:   692 | Loss: 5.2414 | lr = 0.000130 | Norm = 0.386378 | Time: 35.3948sec | Token/sec: 1851.569
    Step:   693 | Loss: 5.2970 | lr = 0.000129 | Norm = 0.397836 | Time: 35.1720sec | Token/sec: 1863.299
    Step:   694 | Loss: 5.1695 | lr = 0.000128 | Norm = 0.397742 | Time: 35.0616sec | Token/sec: 1869.169
    Step:   695 | Loss: 5.2273 | lr = 0.000128 | Norm = 0.387509 | Time: 34.9811sec | Token/sec: 1873.471
    Step:   696 | Loss: 5.1171 | lr = 0.000127 | Norm = 0.369244 | Time: 35.1142sec | Token/sec: 1866.366
    Step:   697 | Loss: 5.1251 | lr = 0.000126 | Norm = 0.373152 | Time: 35.2460sec | Token/sec: 1859.388
    Step:   698 | Loss: 5.2337 | lr = 0.000126 | Norm = 0.350073 | Time: 35.3242sec | Token/sec: 1855.270
    Step:   699 | Loss: 5.2186 | lr = 0.000125 | Norm = 0.439916 | Time: 35.4582sec | Token/sec: 1848.261
    Validation loss: 5.2942
    Rank: 1 | Sample1: I am crazy man, the son of the Lord would leave the son’s house in the heart to the Lord.”
    “I didn””’d all
    Rank: 1 | Sample2: I am crazy man, was more.
    One person was the biggest one of three were only found that an adult or less, a few of whom we should want better for those to see, and
    Rank: 1 | Sample3: I am crazy man, one, the most important of which’s not been.
    It does not know that I had been trying to know,’s what could be, the first
    Rank: 0 | Sample1: I am crazy man, and he could also put himself out of a way of the woman to keep his hands back in his day. However he went to me by a friend of a woman as soon
    Rank: 0 | Sample2: I am crazy man, we’re you are going to say him to get his day-to her and no kind with my own or her.
    Some of those men are born because they
    Rank: 0 | Sample3: I am crazy man, it might be a big kind of life. There are numerous people who are no sense to be the same concern that it could be any thing.
    There is no need to
    HellaSwag accuracy: 2365/10042=0.2355
    Step:   700 | Loss: 5.1923 | lr = 0.000125 | Norm = 0.422804 | Time: 366.7892sec | Token/sec: 178.675
    Step:   701 | Loss: 5.2306 | lr = 0.000124 | Norm = 0.437792 | Time: 35.3372sec | Token/sec: 1854.590
    Step:   702 | Loss: 5.2149 | lr = 0.000123 | Norm = 0.452970 | Time: 35.4474sec | Token/sec: 1848.825
    Step:   703 | Loss: 5.2496 | lr = 0.000123 | Norm = 0.358967 | Time: 35.4357sec | Token/sec: 1849.436
    Step:   704 | Loss: 5.1438 | lr = 0.000122 | Norm = 0.433129 | Time: 35.4376sec | Token/sec: 1849.334
    Step:   705 | Loss: 5.2339 | lr = 0.000121 | Norm = 0.343766 | Time: 35.4512sec | Token/sec: 1848.627
    Step:   706 | Loss: 5.1596 | lr = 0.000121 | Norm = 0.447369 | Time: 35.4315sec | Token/sec: 1849.654
    Step:   707 | Loss: 5.1856 | lr = 0.000120 | Norm = 0.394090 | Time: 35.4509sec | Token/sec: 1848.644
    Step:   708 | Loss: 5.1981 | lr = 0.000120 | Norm = 0.383049 | Time: 35.4303sec | Token/sec: 1849.714
    Step:   709 | Loss: 5.1812 | lr = 0.000119 | Norm = 0.388142 | Time: 35.4101sec | Token/sec: 1850.774
    Step:   710 | Loss: 5.3461 | lr = 0.000118 | Norm = 0.433222 | Time: 35.4716sec | Token/sec: 1847.563
    Step:   711 | Loss: 5.2619 | lr = 0.000118 | Norm = 0.373514 | Time: 35.4250sec | Token/sec: 1849.992
    Step:   712 | Loss: 5.1954 | lr = 0.000117 | Norm = 0.378721 | Time: 35.4222sec | Token/sec: 1850.139
    Step:   713 | Loss: 5.1770 | lr = 0.000117 | Norm = 0.398193 | Time: 35.4496sec | Token/sec: 1848.709
    Step:   714 | Loss: 5.1339 | lr = 0.000116 | Norm = 0.408695 | Time: 35.4027sec | Token/sec: 1851.157
    Step:   715 | Loss: 5.1365 | lr = 0.000116 | Norm = 0.357203 | Time: 35.4199sec | Token/sec: 1850.260
    Step:   716 | Loss: 5.1827 | lr = 0.000115 | Norm = 0.413731 | Time: 35.4269sec | Token/sec: 1849.893
    Step:   717 | Loss: 5.0734 | lr = 0.000114 | Norm = 0.390286 | Time: 35.4142sec | Token/sec: 1850.559
    Step:   718 | Loss: 5.1804 | lr = 0.000114 | Norm = 0.381724 | Time: 35.3618sec | Token/sec: 1853.301
    Step:   719 | Loss: 5.1355 | lr = 0.000113 | Norm = 0.404123 | Time: 35.4214sec | Token/sec: 1850.180
    Step:   720 | Loss: 5.1094 | lr = 0.000113 | Norm = 0.357075 | Time: 35.3972sec | Token/sec: 1851.448
    Step:   721 | Loss: 5.0419 | lr = 0.000112 | Norm = 0.411089 | Time: 35.4442sec | Token/sec: 1848.992
    Step:   722 | Loss: 5.1176 | lr = 0.000112 | Norm = 0.321345 | Time: 35.4557sec | Token/sec: 1848.389
    Step:   723 | Loss: 5.2428 | lr = 0.000111 | Norm = 0.401330 | Time: 35.5071sec | Token/sec: 1845.715
    Step:   724 | Loss: 5.1725 | lr = 0.000110 | Norm = 0.372746 | Time: 35.1514sec | Token/sec: 1864.392
    Step:   725 | Loss: 5.1517 | lr = 0.000110 | Norm = 0.391166 | Time: 34.9815sec | Token/sec: 1873.449
    Step:   726 | Loss: 5.1820 | lr = 0.000109 | Norm = 0.362201 | Time: 35.1454sec | Token/sec: 1864.708
    Step:   727 | Loss: 5.1489 | lr = 0.000109 | Norm = 0.347054 | Time: 35.2586sec | Token/sec: 1858.722
    Step:   728 | Loss: 5.2020 | lr = 0.000108 | Norm = 0.335254 | Time: 35.3528sec | Token/sec: 1853.769
    Step:   729 | Loss: 5.1456 | lr = 0.000108 | Norm = 0.368749 | Time: 35.4754sec | Token/sec: 1847.364
    Step:   730 | Loss: 5.1450 | lr = 0.000107 | Norm = 0.358664 | Time: 35.4459sec | Token/sec: 1848.902
    Step:   731 | Loss: 5.1764 | lr = 0.000107 | Norm = 0.392971 | Time: 35.4738sec | Token/sec: 1847.446
    Step:   732 | Loss: 5.1262 | lr = 0.000106 | Norm = 0.370954 | Time: 35.0136sec | Token/sec: 1871.729
    Step:   733 | Loss: 5.1892 | lr = 0.000106 | Norm = 0.419437 | Time: 35.5375sec | Token/sec: 1844.138
    Step:   734 | Loss: 5.1701 | lr = 0.000105 | Norm = 0.333926 | Time: 34.9656sec | Token/sec: 1874.297
    Step:   735 | Loss: 5.1833 | lr = 0.000105 | Norm = 0.398929 | Time: 35.3554sec | Token/sec: 1853.636
    Step:   736 | Loss: 5.2440 | lr = 0.000104 | Norm = 0.395515 | Time: 35.5142sec | Token/sec: 1845.348
    Step:   737 | Loss: 5.1688 | lr = 0.000103 | Norm = 0.364640 | Time: 35.3773sec | Token/sec: 1852.489
    Step:   738 | Loss: 5.2012 | lr = 0.000103 | Norm = 0.440490 | Time: 35.1800sec | Token/sec: 1862.876
    Step:   739 | Loss: 5.1446 | lr = 0.000102 | Norm = 0.382127 | Time: 34.9985sec | Token/sec: 1872.536
    Step:   740 | Loss: 5.1691 | lr = 0.000102 | Norm = 0.469940 | Time: 34.9726sec | Token/sec: 1873.926
    Step:   741 | Loss: 5.2801 | lr = 0.000101 | Norm = 0.362353 | Time: 35.1502sec | Token/sec: 1864.458
    Step:   742 | Loss: 5.3096 | lr = 0.000101 | Norm = 0.423819 | Time: 35.2824sec | Token/sec: 1857.472
    Step:   743 | Loss: 5.3005 | lr = 0.000100 | Norm = 0.516711 | Time: 35.3812sec | Token/sec: 1852.281
    Step:   744 | Loss: 5.2757 | lr = 0.000100 | Norm = 0.412978 | Time: 35.4126sec | Token/sec: 1850.640
    Step:   745 | Loss: 5.2829 | lr = 0.000099 | Norm = 0.369077 | Time: 35.4334sec | Token/sec: 1849.553
    Step:   746 | Loss: 5.3046 | lr = 0.000099 | Norm = 0.394617 | Time: 35.4512sec | Token/sec: 1848.625
    Step:   747 | Loss: 5.2971 | lr = 0.000098 | Norm = 0.408731 | Time: 35.4894sec | Token/sec: 1846.638
    Step:   748 | Loss: 5.2978 | lr = 0.000098 | Norm = 0.350578 | Time: 35.4321sec | Token/sec: 1849.621
    Step:   749 | Loss: 5.3587 | lr = 0.000097 | Norm = 0.444008 | Time: 35.4224sec | Token/sec: 1850.130
    Step:   750 | Loss: 5.4031 | lr = 0.000097 | Norm = 0.381872 | Time: 35.4627sec | Token/sec: 1848.027
    Step:   751 | Loss: 5.2949 | lr = 0.000096 | Norm = 0.412211 | Time: 35.4473sec | Token/sec: 1848.828
    Step:   752 | Loss: 5.2383 | lr = 0.000096 | Norm = 0.419437 | Time: 35.3945sec | Token/sec: 1851.587
    Step:   753 | Loss: 5.2909 | lr = 0.000096 | Norm = 0.369070 | Time: 35.4495sec | Token/sec: 1848.712
    Step:   754 | Loss: 5.2711 | lr = 0.000095 | Norm = 0.431610 | Time: 35.3143sec | Token/sec: 1855.793
    Step:   755 | Loss: 5.3939 | lr = 0.000095 | Norm = 0.461903 | Time: 35.1558sec | Token/sec: 1864.159
    Step:   756 | Loss: 5.2675 | lr = 0.000094 | Norm = 0.470137 | Time: 34.9854sec | Token/sec: 1873.241
    Step:   757 | Loss: 5.3098 | lr = 0.000094 | Norm = 0.420521 | Time: 34.9415sec | Token/sec: 1875.590
    Step:   758 | Loss: 5.2695 | lr = 0.000093 | Norm = 0.468632 | Time: 35.1883sec | Token/sec: 1862.440
    Step:   759 | Loss: 5.3371 | lr = 0.000093 | Norm = 0.459239 | Time: 35.3015sec | Token/sec: 1856.464
    Step:   760 | Loss: 5.3587 | lr = 0.000092 | Norm = 0.433587 | Time: 35.3222sec | Token/sec: 1855.379
    Step:   761 | Loss: 5.3596 | lr = 0.000092 | Norm = 0.402443 | Time: 35.4441sec | Token/sec: 1848.994
    Step:   762 | Loss: 5.3208 | lr = 0.000091 | Norm = 0.469577 | Time: 35.3753sec | Token/sec: 1852.591
    Step:   763 | Loss: 5.2958 | lr = 0.000091 | Norm = 0.386633 | Time: 35.4238sec | Token/sec: 1850.053
    Step:   764 | Loss: 5.3176 | lr = 0.000091 | Norm = 0.454380 | Time: 35.4563sec | Token/sec: 1848.362
    Step:   765 | Loss: 5.2679 | lr = 0.000090 | Norm = 0.381783 | Time: 35.4463sec | Token/sec: 1848.882
    Step:   766 | Loss: 5.2250 | lr = 0.000090 | Norm = 0.404871 | Time: 35.4695sec | Token/sec: 1847.672
    Step:   767 | Loss: 5.3212 | lr = 0.000089 | Norm = 0.392346 | Time: 35.2156sec | Token/sec: 1860.991
    Step:   768 | Loss: 5.3763 | lr = 0.000089 | Norm = 0.402203 | Time: 35.0072sec | Token/sec: 1872.072
    Step:   769 | Loss: 5.3645 | lr = 0.000088 | Norm = 0.480916 | Time: 34.9983sec | Token/sec: 1872.548
    Step:   770 | Loss: 5.3081 | lr = 0.000088 | Norm = 0.370826 | Time: 35.0983sec | Token/sec: 1867.214
    Step:   771 | Loss: 5.2469 | lr = 0.000088 | Norm = 0.420210 | Time: 35.3706sec | Token/sec: 1852.839
    Step:   772 | Loss: 5.3650 | lr = 0.000087 | Norm = 0.403050 | Time: 35.4620sec | Token/sec: 1848.063
    Step:   773 | Loss: 5.3092 | lr = 0.000087 | Norm = 0.423227 | Time: 35.4712sec | Token/sec: 1847.584
    Step:   774 | Loss: 5.2758 | lr = 0.000086 | Norm = 0.409706 | Time: 35.4269sec | Token/sec: 1849.893
    Step:   775 | Loss: 5.3610 | lr = 0.000086 | Norm = 0.527803 | Time: 35.4883sec | Token/sec: 1846.695
    Step:   776 | Loss: 5.3653 | lr = 0.000085 | Norm = 0.470366 | Time: 35.4527sec | Token/sec: 1848.549
    Step:   777 | Loss: 5.2047 | lr = 0.000085 | Norm = 0.517994 | Time: 35.4303sec | Token/sec: 1849.715
    Step:   778 | Loss: 5.2839 | lr = 0.000085 | Norm = 0.410426 | Time: 35.4523sec | Token/sec: 1848.570
    Step:   779 | Loss: 5.2664 | lr = 0.000084 | Norm = 0.412926 | Time: 35.4558sec | Token/sec: 1848.386
    Step:   780 | Loss: 5.2724 | lr = 0.000084 | Norm = 0.447966 | Time: 35.4835sec | Token/sec: 1846.942
    Step:   781 | Loss: 5.2791 | lr = 0.000083 | Norm = 0.452582 | Time: 35.4797sec | Token/sec: 1847.138
    Step:   782 | Loss: 5.1536 | lr = 0.000083 | Norm = 0.420647 | Time: 35.2111sec | Token/sec: 1861.232
    Step:   783 | Loss: 5.2995 | lr = 0.000083 | Norm = 0.428380 | Time: 35.0678sec | Token/sec: 1868.836
    Step:   784 | Loss: 5.3513 | lr = 0.000082 | Norm = 0.427673 | Time: 35.0295sec | Token/sec: 1870.882
    Step:   785 | Loss: 5.2678 | lr = 0.000082 | Norm = 0.456470 | Time: 34.9971sec | Token/sec: 1872.611
    Step:   786 | Loss: 5.4085 | lr = 0.000082 | Norm = 0.429285 | Time: 34.9966sec | Token/sec: 1872.638
    Step:   787 | Loss: 5.6289 | lr = 0.000081 | Norm = 0.591968 | Time: 35.1622sec | Token/sec: 1863.817
    Step:   788 | Loss: 5.2928 | lr = 0.000081 | Norm = 0.599120 | Time: 35.2691sec | Token/sec: 1858.171
    Step:   789 | Loss: 5.2433 | lr = 0.000080 | Norm = 0.472281 | Time: 35.3049sec | Token/sec: 1856.286
    Step:   790 | Loss: 5.3351 | lr = 0.000080 | Norm = 0.423866 | Time: 35.3910sec | Token/sec: 1851.770
    Step:   791 | Loss: 5.2548 | lr = 0.000080 | Norm = 0.503042 | Time: 35.4269sec | Token/sec: 1849.891
    Step:   792 | Loss: 5.3860 | lr = 0.000079 | Norm = 0.442509 | Time: 35.4273sec | Token/sec: 1849.874
    Step:   793 | Loss: 5.2270 | lr = 0.000079 | Norm = 0.397444 | Time: 35.4366sec | Token/sec: 1849.387
    Step:   794 | Loss: 5.3475 | lr = 0.000079 | Norm = 0.410025 | Time: 35.4496sec | Token/sec: 1848.711
    Step:   795 | Loss: 5.3536 | lr = 0.000078 | Norm = 0.380340 | Time: 35.4273sec | Token/sec: 1849.872
    Step:   796 | Loss: 5.2728 | lr = 0.000078 | Norm = 0.413914 | Time: 35.4492sec | Token/sec: 1848.732
    Step:   797 | Loss: 5.2960 | lr = 0.000078 | Norm = 0.380279 | Time: 35.5153sec | Token/sec: 1845.288
    Step:   798 | Loss: 5.3426 | lr = 0.000077 | Norm = 0.461101 | Time: 35.4052sec | Token/sec: 1851.029
    Step:   799 | Loss: 5.2125 | lr = 0.000077 | Norm = 0.407054 | Time: 35.1749sec | Token/sec: 1863.144
    Validation loss: 5.2150
    Rank: 1 | Sample1: I am crazy man, he said, He has always learned himself for one that one is now. And in the same person would have seen having the first person that it’s going out of
    Rank: 1 | Sample2: I am crazy man, and all are the kind of thought to do so, but also the earth is a very important, and the only way of the universe. I believe that it seems to think
    Rank: 1 | Sample3: I am crazy man, then by a couple or four children in the middle and half-two-hour time of course.”
    “It’s the real-world job is
    Rank: 0 | Sample1: I am crazy man, it was a family. This is so-called the very part of the second world that it was in the state itself.
    This type of events which has in its universe
    Rank: 0 | Sample2: I am crazy man, that it wasn did have done it. He said, and she said she would be so. He should be able to move for the ground so he gave him as to the
    Rank: 0 | Sample3: I am crazy man, he took a lot of things and his heart.” He was known as the first man in a story of the book. He is given much of his life and he
    HellaSwag accuracy: 2361/10042=0.2351
    Step:   800 | Loss: 5.2141 | lr = 0.000077 | Norm = 0.387348 | Time: 366.6191sec | Token/sec: 178.758
    Step:   801 | Loss: 5.2702 | lr = 0.000076 | Norm = 0.398715 | Time: 35.3919sec | Token/sec: 1851.721
    Step:   802 | Loss: 5.2920 | lr = 0.000076 | Norm = 0.371968 | Time: 35.3387sec | Token/sec: 1854.509
    Step:   803 | Loss: 5.3086 | lr = 0.000076 | Norm = 0.384997 | Time: 35.1019sec | Token/sec: 1867.019
    Step:   804 | Loss: 5.2665 | lr = 0.000075 | Norm = 0.376513 | Time: 35.1705sec | Token/sec: 1863.382
    Step:   805 | Loss: 5.2274 | lr = 0.000075 | Norm = 0.355373 | Time: 35.2822sec | Token/sec: 1857.483
    Step:   806 | Loss: 5.2585 | lr = 0.000075 | Norm = 0.355989 | Time: 35.3985sec | Token/sec: 1851.379
    Step:   807 | Loss: 5.2424 | lr = 0.000074 | Norm = 0.413599 | Time: 35.5038sec | Token/sec: 1845.885
    Step:   808 | Loss: 5.1684 | lr = 0.000074 | Norm = 0.327766 | Time: 35.2761sec | Token/sec: 1857.800
    Step:   809 | Loss: 5.2148 | lr = 0.000074 | Norm = 0.384880 | Time: 35.2522sec | Token/sec: 1859.059
    Step:   810 | Loss: 5.3005 | lr = 0.000074 | Norm = 0.362202 | Time: 35.0990sec | Token/sec: 1867.175
    Step:   811 | Loss: 5.2070 | lr = 0.000073 | Norm = 0.356014 | Time: 35.1384sec | Token/sec: 1865.083
    Step:   812 | Loss: 5.2634 | lr = 0.000073 | Norm = 0.375770 | Time: 34.9933sec | Token/sec: 1872.816
    Step:   813 | Loss: 5.3685 | lr = 0.000073 | Norm = 0.454657 | Time: 34.9807sec | Token/sec: 1873.490
    Step:   814 | Loss: 5.3014 | lr = 0.000072 | Norm = 0.397454 | Time: 35.1563sec | Token/sec: 1864.134
    Step:   815 | Loss: 5.1726 | lr = 0.000072 | Norm = 0.403638 | Time: 35.3128sec | Token/sec: 1855.872
    Step:   816 | Loss: 5.2220 | lr = 0.000072 | Norm = 0.404621 | Time: 35.3929sec | Token/sec: 1851.673
    Step:   817 | Loss: 5.2620 | lr = 0.000072 | Norm = 0.383311 | Time: 35.4487sec | Token/sec: 1848.756
    Step:   818 | Loss: 5.4042 | lr = 0.000071 | Norm = 0.426844 | Time: 35.4436sec | Token/sec: 1849.024
    Step:   819 | Loss: 5.2055 | lr = 0.000071 | Norm = 0.395394 | Time: 35.4484sec | Token/sec: 1848.771
    Step:   820 | Loss: 5.1954 | lr = 0.000071 | Norm = 0.378693 | Time: 35.2073sec | Token/sec: 1861.433
    Step:   821 | Loss: 5.2184 | lr = 0.000070 | Norm = 0.387631 | Time: 35.1373sec | Token/sec: 1865.141
    Step:   822 | Loss: 5.1685 | lr = 0.000070 | Norm = 0.412293 | Time: 35.0723sec | Token/sec: 1868.598
    Step:   823 | Loss: 5.2251 | lr = 0.000070 | Norm = 0.351814 | Time: 35.3178sec | Token/sec: 1855.609
    Step:   824 | Loss: 5.2165 | lr = 0.000070 | Norm = 0.395263 | Time: 35.4560sec | Token/sec: 1848.375
    Step:   825 | Loss: 5.4251 | lr = 0.000069 | Norm = 0.850357 | Time: 35.5029sec | Token/sec: 1845.935
    Step:   826 | Loss: 5.3193 | lr = 0.000069 | Norm = 0.437178 | Time: 35.3577sec | Token/sec: 1853.514
    Step:   827 | Loss: 5.2297 | lr = 0.000069 | Norm = 0.473441 | Time: 35.2003sec | Token/sec: 1861.803
    Step:   828 | Loss: 5.2621 | lr = 0.000069 | Norm = 0.435624 | Time: 35.1493sec | Token/sec: 1864.505
    Step:   829 | Loss: 5.2691 | lr = 0.000068 | Norm = 0.460462 | Time: 35.0638sec | Token/sec: 1869.053
    Step:   830 | Loss: 5.2986 | lr = 0.000068 | Norm = 0.506314 | Time: 35.2467sec | Token/sec: 1859.349
    Step:   831 | Loss: 5.3005 | lr = 0.000068 | Norm = 0.409673 | Time: 35.4610sec | Token/sec: 1848.117
    Step:   832 | Loss: 5.2715 | lr = 0.000068 | Norm = 0.429263 | Time: 35.3640sec | Token/sec: 1853.183
    Step:   833 | Loss: 5.2245 | lr = 0.000068 | Norm = 0.357523 | Time: 35.4536sec | Token/sec: 1848.503
    Step:   834 | Loss: 5.1943 | lr = 0.000067 | Norm = 0.412157 | Time: 35.4315sec | Token/sec: 1849.656
    Step:   835 | Loss: 5.2311 | lr = 0.000067 | Norm = 0.368257 | Time: 35.4244sec | Token/sec: 1850.025
    Step:   836 | Loss: 5.3841 | lr = 0.000067 | Norm = 0.540103 | Time: 35.4538sec | Token/sec: 1848.493
    Step:   837 | Loss: 5.3534 | lr = 0.000067 | Norm = 0.486789 | Time: 35.1693sec | Token/sec: 1863.441
    Step:   838 | Loss: 5.8255 | lr = 0.000066 | Norm = 0.686548 | Time: 35.2420sec | Token/sec: 1859.600
    Step:   839 | Loss: 5.1775 | lr = 0.000066 | Norm = 0.529349 | Time: 35.3422sec | Token/sec: 1854.327
    Step:   840 | Loss: 4.9930 | lr = 0.000066 | Norm = 1.578941 | Time: 35.3228sec | Token/sec: 1855.344
    Step:   841 | Loss: 5.1482 | lr = 0.000066 | Norm = 0.456984 | Time: 35.4067sec | Token/sec: 1850.950
    Step:   842 | Loss: 5.1781 | lr = 0.000066 | Norm = 0.433909 | Time: 35.4869sec | Token/sec: 1846.769
    Step:   843 | Loss: 5.2432 | lr = 0.000065 | Norm = 0.359837 | Time: 35.4899sec | Token/sec: 1846.612
    Step:   844 | Loss: 5.2317 | lr = 0.000065 | Norm = 0.392715 | Time: 35.2644sec | Token/sec: 1858.419
    Step:   845 | Loss: 5.2199 | lr = 0.000065 | Norm = 0.455279 | Time: 35.1867sec | Token/sec: 1862.521
    Step:   846 | Loss: 5.2586 | lr = 0.000065 | Norm = 0.357557 | Time: 35.1712sec | Token/sec: 1863.343
    Step:   847 | Loss: 5.1941 | lr = 0.000065 | Norm = 0.392712 | Time: 35.2447sec | Token/sec: 1859.454
    Step:   848 | Loss: 5.1821 | lr = 0.000065 | Norm = 0.442341 | Time: 35.3992sec | Token/sec: 1851.340
    Step:   849 | Loss: 5.1794 | lr = 0.000064 | Norm = 0.416205 | Time: 35.3923sec | Token/sec: 1851.704
    Step:   850 | Loss: 5.1801 | lr = 0.000064 | Norm = 0.468592 | Time: 35.4171sec | Token/sec: 1850.404
    Step:   851 | Loss: 5.2704 | lr = 0.000064 | Norm = 0.361098 | Time: 35.4660sec | Token/sec: 1847.852
    Step:   852 | Loss: 5.2523 | lr = 0.000064 | Norm = 0.434952 | Time: 35.4796sec | Token/sec: 1847.147
    Step:   853 | Loss: 5.2564 | lr = 0.000064 | Norm = 0.456915 | Time: 35.4713sec | Token/sec: 1847.577
    Step:   854 | Loss: 5.1082 | lr = 0.000064 | Norm = 0.409979 | Time: 35.5300sec | Token/sec: 1844.525
    Step:   855 | Loss: 5.1665 | lr = 0.000063 | Norm = 0.383709 | Time: 35.3153sec | Token/sec: 1855.741
    Step:   856 | Loss: 5.1358 | lr = 0.000063 | Norm = 0.384196 | Time: 35.1206sec | Token/sec: 1866.029
    Step:   857 | Loss: 5.4028 | lr = 0.000063 | Norm = 0.832329 | Time: 35.0497sec | Token/sec: 1869.800
    Step:   858 | Loss: 5.1230 | lr = 0.000063 | Norm = 0.383447 | Time: 35.0317sec | Token/sec: 1870.763
    Step:   859 | Loss: 5.0856 | lr = 0.000063 | Norm = 0.376952 | Time: 34.9765sec | Token/sec: 1873.716
    Step:   860 | Loss: 5.1295 | lr = 0.000063 | Norm = 0.410767 | Time: 35.1572sec | Token/sec: 1864.084
    Step:   861 | Loss: 5.3246 | lr = 0.000063 | Norm = 0.520110 | Time: 35.3453sec | Token/sec: 1854.165
    Step:   862 | Loss: 5.1365 | lr = 0.000062 | Norm = 0.464502 | Time: 35.4076sec | Token/sec: 1850.903
    Step:   863 | Loss: 5.1869 | lr = 0.000062 | Norm = 0.420701 | Time: 35.4695sec | Token/sec: 1847.672
    Step:   864 | Loss: 5.0718 | lr = 0.000062 | Norm = 0.395151 | Time: 35.2568sec | Token/sec: 1858.816
    Step:   865 | Loss: 5.1414 | lr = 0.000062 | Norm = 0.481723 | Time: 35.2458sec | Token/sec: 1859.399
    Step:   866 | Loss: 5.1143 | lr = 0.000062 | Norm = 0.371993 | Time: 35.4823sec | Token/sec: 1847.006
    Step:   867 | Loss: 5.1730 | lr = 0.000062 | Norm = 0.457056 | Time: 35.3989sec | Token/sec: 1851.359
    Step:   868 | Loss: 5.1825 | lr = 0.000062 | Norm = 0.451551 | Time: 35.4424sec | Token/sec: 1849.083
    Step:   869 | Loss: 5.0448 | lr = 0.000062 | Norm = 0.419530 | Time: 35.4832sec | Token/sec: 1846.960
    Step:   870 | Loss: 5.1081 | lr = 0.000062 | Norm = 0.368784 | Time: 35.3952sec | Token/sec: 1851.550
    Step:   871 | Loss: 5.1450 | lr = 0.000061 | Norm = 0.391148 | Time: 35.3145sec | Token/sec: 1855.781
    Step:   872 | Loss: 5.0799 | lr = 0.000061 | Norm = 0.391695 | Time: 35.1685sec | Token/sec: 1863.488
    Step:   873 | Loss: 5.0786 | lr = 0.000061 | Norm = 0.413218 | Time: 35.1339sec | Token/sec: 1865.320
    Step:   874 | Loss: 5.1675 | lr = 0.000061 | Norm = 0.345128 | Time: 35.0310sec | Token/sec: 1870.798
    Step:   875 | Loss: 5.1098 | lr = 0.000061 | Norm = 0.403608 | Time: 35.1461sec | Token/sec: 1864.672
    Step:   876 | Loss: 5.1576 | lr = 0.000061 | Norm = 0.374449 | Time: 35.2389sec | Token/sec: 1859.766
    Step:   877 | Loss: 5.1214 | lr = 0.000061 | Norm = 0.390969 | Time: 35.3248sec | Token/sec: 1855.239
    Step:   878 | Loss: 5.0923 | lr = 0.000061 | Norm = 0.355885 | Time: 35.4068sec | Token/sec: 1850.943
    Step:   879 | Loss: 5.0332 | lr = 0.000061 | Norm = 0.347473 | Time: 35.4717sec | Token/sec: 1847.556
    Step:   880 | Loss: 5.1265 | lr = 0.000061 | Norm = 0.340790 | Time: 35.5315sec | Token/sec: 1844.450
    Step:   881 | Loss: 5.0948 | lr = 0.000061 | Norm = 0.375414 | Time: 35.4476sec | Token/sec: 1848.814
    Step:   882 | Loss: 5.0482 | lr = 0.000061 | Norm = 0.459216 | Time: 35.1894sec | Token/sec: 1862.378
    Step:   883 | Loss: 5.2095 | lr = 0.000060 | Norm = 0.417677 | Time: 35.1340sec | Token/sec: 1865.314
    Step:   884 | Loss: 5.2225 | lr = 0.000060 | Norm = 0.481615 | Time: 35.0531sec | Token/sec: 1869.619
    Step:   885 | Loss: 5.2012 | lr = 0.000060 | Norm = 0.381188 | Time: 35.1593sec | Token/sec: 1863.976
    Step:   886 | Loss: 5.0893 | lr = 0.000060 | Norm = 0.333016 | Time: 35.2550sec | Token/sec: 1858.912
    Step:   887 | Loss: 5.1232 | lr = 0.000060 | Norm = 0.370319 | Time: 35.3168sec | Token/sec: 1855.659
    Step:   888 | Loss: 5.2138 | lr = 0.000060 | Norm = 0.358090 | Time: 35.4297sec | Token/sec: 1849.746
    Step:   889 | Loss: 5.1117 | lr = 0.000060 | Norm = 0.415490 | Time: 35.4440sec | Token/sec: 1849.003
    Step:   890 | Loss: 5.1683 | lr = 0.000060 | Norm = 0.363379 | Time: 35.4004sec | Token/sec: 1851.276
    Step:   891 | Loss: 5.0449 | lr = 0.000060 | Norm = 0.363353 | Time: 35.4864sec | Token/sec: 1846.791
    Step:   892 | Loss: 5.0749 | lr = 0.000060 | Norm = 0.598953 | Time: 35.4125sec | Token/sec: 1850.648
    Step:   893 | Loss: 5.1814 | lr = 0.000060 | Norm = 0.349627 | Time: 35.4685sec | Token/sec: 1847.723
    Step:   894 | Loss: 5.2793 | lr = 0.000060 | Norm = 0.392088 | Time: 35.4498sec | Token/sec: 1848.700
    Step:   895 | Loss: 5.3025 | lr = 0.000060 | Norm = 0.454596 | Time: 35.5097sec | Token/sec: 1845.582
    Step:   896 | Loss: 5.1228 | lr = 0.000060 | Norm = 0.374708 | Time: 35.4419sec | Token/sec: 1849.112
    Step:   897 | Loss: 5.2297 | lr = 0.000060 | Norm = 0.493139 | Time: 35.2546sec | Token/sec: 1858.933
    Step:   898 | Loss: 5.0923 | lr = 0.000060 | Norm = 0.385098 | Time: 35.1530sec | Token/sec: 1864.306
    Step:   899 | Loss: 5.1365 | lr = 0.000060 | Norm = 0.391660 | Time: 35.0140sec | Token/sec: 1871.706
    Validation loss: 5.1678
    Rank: 1 | Sample1: I am crazy man, ‘I cannot be too a girl with you know what you never want to do?’
    A very good example of writing about those children have a little about something
    Rank: 1 | Sample2: I am crazy man, was all the day to say, how you have no point!
    The only way to get out.
    There are no reason for you on, even if you are looking
    Rank: 1 | Sample3: I am crazy man, I do that you don’t have to say the things that have done to do the words, and therefore I shall never think with my body. Let’s
    Rank: 0 | Sample1: I am crazy man, and said, but if every one one of the days of the other. I will see if I have already passed on to the following my house. I wanted me. I
    Rank: 0 | Sample2: I am crazy man, though he never had never told his wife's. He had given the first wife, and we are of a better job, and this week, his father he was a husband
    Rank: 0 | Sample3: I am crazy man, “Rohah,”
    I would I did not love. The first time of Jesus, I went with His death at times the time you have a good
    HellaSwag accuracy: 2335/10042=0.2325
    Step:   900 | Loss: 5.0305 | lr = 0.000060 | Norm = 0.376673 | Time: 367.3904sec | Token/sec: 178.382
    Step:   901 | Loss: 5.0857 | lr = 0.000060 | Norm = 0.414886 | Time: 35.0075sec | Token/sec: 1872.054
    Step:   902 | Loss: 5.0987 | lr = 0.000060 | Norm = 0.351471 | Time: 35.2094sec | Token/sec: 1861.319
    Step:   903 | Loss: 5.0085 | lr = 0.000060 | Norm = 0.372961 | Time: 35.3570sec | Token/sec: 1853.553
    Step:   904 | Loss: 5.0054 | lr = 0.000060 | Norm = 0.367226 | Time: 35.4706sec | Token/sec: 1847.614
    Step:   905 | Loss: 5.0589 | lr = 0.000060 | Norm = 0.357379 | Time: 35.5154sec | Token/sec: 1845.286
    Step:   906 | Loss: 5.0517 | lr = 0.000060 | Norm = 0.424922 | Time: 35.5120sec | Token/sec: 1845.462
    Step:   907 | Loss: 5.0672 | lr = 0.000060 | Norm = 0.381245 | Time: 35.3270sec | Token/sec: 1855.124
    Step:   908 | Loss: 5.1169 | lr = 0.000060 | Norm = 0.383173 | Time: 35.0176sec | Token/sec: 1871.515
    Step:   909 | Loss: 5.1908 | lr = 0.000060 | Norm = 0.369948 | Time: 35.0320sec | Token/sec: 1870.745
    Step:   910 | Loss: 5.1213 | lr = 0.000060 | Norm = 0.397943 | Time: 35.0587sec | Token/sec: 1869.324
    Step:   911 | Loss: 5.0420 | lr = 0.000060 | Norm = 0.385681 | Time: 35.2646sec | Token/sec: 1858.405
    Step:   912 | Loss: 4.9773 | lr = 0.000060 | Norm = 0.384267 | Time: 35.4333sec | Token/sec: 1849.560
    Step:   913 | Loss: 5.0626 | lr = 0.000060 | Norm = 0.353818 | Time: 35.5007sec | Token/sec: 1846.047
    Step:   914 | Loss: 4.9864 | lr = 0.000060 | Norm = 0.380992 | Time: 35.4884sec | Token/sec: 1846.688
    Step:   915 | Loss: 5.0122 | lr = 0.000060 | Norm = 0.629941 | Time: 35.4964sec | Token/sec: 1846.269
    Step:   916 | Loss: 4.9033 | lr = 0.000060 | Norm = 0.461778 | Time: 35.3667sec | Token/sec: 1853.042
    Step:   917 | Loss: 5.1488 | lr = 0.000060 | Norm = 0.393929 | Time: 35.1400sec | Token/sec: 1864.999
    Step:   918 | Loss: 5.1483 | lr = 0.000060 | Norm = 0.450262 | Time: 35.0119sec | Token/sec: 1871.821
    Step:   919 | Loss: 5.0435 | lr = 0.000060 | Norm = 0.375819 | Time: 34.9754sec | Token/sec: 1873.774
    Step:   920 | Loss: 5.1336 | lr = 0.000060 | Norm = 0.407187 | Time: 35.0956sec | Token/sec: 1867.358
    Step:   921 | Loss: 5.0746 | lr = 0.000060 | Norm = 0.377995 | Time: 35.3090sec | Token/sec: 1856.069
    Step:   922 | Loss: 5.0906 | lr = 0.000060 | Norm = 0.409044 | Time: 35.4916sec | Token/sec: 1846.523
    Step:   923 | Loss: 5.0349 | lr = 0.000060 | Norm = 0.360422 | Time: 35.4429sec | Token/sec: 1849.058
    Step:   924 | Loss: 4.9918 | lr = 0.000060 | Norm = 0.386741 | Time: 35.5547sec | Token/sec: 1843.246
    Step:   925 | Loss: 5.1909 | lr = 0.000060 | Norm = 0.413470 | Time: 35.4436sec | Token/sec: 1849.024
    Step:   926 | Loss: 5.0254 | lr = 0.000060 | Norm = 0.359500 | Time: 35.1757sec | Token/sec: 1863.106
    Step:   927 | Loss: 5.0224 | lr = 0.000060 | Norm = 0.410255 | Time: 35.1591sec | Token/sec: 1863.984
    Step:   928 | Loss: 5.1049 | lr = 0.000060 | Norm = 0.430188 | Time: 35.0552sec | Token/sec: 1869.507
    Step:   929 | Loss: 5.0750 | lr = 0.000060 | Norm = 0.552421 | Time: 35.2559sec | Token/sec: 1858.869
    Step:   930 | Loss: 5.1498 | lr = 0.000060 | Norm = 0.564638 | Time: 35.3439sec | Token/sec: 1854.238
    Step:   931 | Loss: 4.9877 | lr = 0.000060 | Norm = 0.586454 | Time: 35.3303sec | Token/sec: 1854.952
    Step:   932 | Loss: 5.0551 | lr = 0.000060 | Norm = 0.456983 | Time: 35.3857sec | Token/sec: 1852.048
    Step:   933 | Loss: 5.0303 | lr = 0.000060 | Norm = 0.419016 | Time: 35.4159sec | Token/sec: 1850.466
    Step:   934 | Loss: 5.0630 | lr = 0.000060 | Norm = 0.476275 | Time: 35.4461sec | Token/sec: 1848.890
    Step:   935 | Loss: 5.0658 | lr = 0.000060 | Norm = 0.425670 | Time: 35.4854sec | Token/sec: 1846.844
    Step:   936 | Loss: 5.0330 | lr = 0.000060 | Norm = 0.466716 | Time: 35.2413sec | Token/sec: 1859.636
    Step:   937 | Loss: 5.2123 | lr = 0.000060 | Norm = 0.409814 | Time: 35.1348sec | Token/sec: 1865.271
    Step:   938 | Loss: 5.0522 | lr = 0.000060 | Norm = 1.868997 | Time: 35.0388sec | Token/sec: 1870.386
    Step:   939 | Loss: 5.1319 | lr = 0.000060 | Norm = 0.563685 | Time: 35.1558sec | Token/sec: 1864.161
    Step:   940 | Loss: 5.0516 | lr = 0.000060 | Norm = 0.409827 | Time: 35.2361sec | Token/sec: 1859.910
    Step:   941 | Loss: 5.0845 | lr = 0.000060 | Norm = 0.516320 | Time: 35.3176sec | Token/sec: 1855.617
    Step:   942 | Loss: 4.9833 | lr = 0.000060 | Norm = 0.493269 | Time: 35.4127sec | Token/sec: 1850.636
    Step:   943 | Loss: 5.0597 | lr = 0.000060 | Norm = 0.495147 | Time: 35.4270sec | Token/sec: 1849.888
    Step:   944 | Loss: 4.9865 | lr = 0.000060 | Norm = 0.615315 | Time: 35.4442sec | Token/sec: 1848.990
    Step:   945 | Loss: 5.0882 | lr = 0.000060 | Norm = 0.431624 | Time: 35.4787sec | Token/sec: 1847.193
    Step:   946 | Loss: 5.0458 | lr = 0.000060 | Norm = 0.456953 | Time: 35.4663sec | Token/sec: 1847.840
    Step:   947 | Loss: 5.0166 | lr = 0.000060 | Norm = 0.440216 | Time: 35.5476sec | Token/sec: 1843.614
    Step:   948 | Loss: 5.0688 | lr = 0.000060 | Norm = 0.402301 | Time: 35.3523sec | Token/sec: 1853.799
    Step:   949 | Loss: 5.0923 | lr = 0.000060 | Norm = 0.456731 | Time: 35.2078sec | Token/sec: 1861.406
    Step:   950 | Loss: 5.2802 | lr = 0.000060 | Norm = 0.489133 | Time: 35.1451sec | Token/sec: 1864.728
    Step:   951 | Loss: 5.1604 | lr = 0.000060 | Norm = 0.428721 | Time: 35.1870sec | Token/sec: 1862.504
    Step:   952 | Loss: 5.2704 | lr = 0.000060 | Norm = 0.385231 | Time: 35.3737sec | Token/sec: 1852.676
    Step:   953 | Loss: 5.2099 | lr = 0.000060 | Norm = 0.419271 | Time: 35.4401sec | Token/sec: 1849.204
    Step:   954 | Loss: 5.2044 | lr = 0.000060 | Norm = 0.403080 | Time: 35.4649sec | Token/sec: 1847.912
    Step:   955 | Loss: 5.2971 | lr = 0.000060 | Norm = 0.395438 | Time: 35.4470sec | Token/sec: 1848.845
    Step:   956 | Loss: 5.1467 | lr = 0.000060 | Norm = 0.420606 | Time: 35.4421sec | Token/sec: 1849.099
    Step:   957 | Loss: 5.2921 | lr = 0.000060 | Norm = 0.447376 | Time: 35.4199sec | Token/sec: 1850.257
    Step:   958 | Loss: 5.2319 | lr = 0.000060 | Norm = 0.352195 | Time: 35.4814sec | Token/sec: 1847.055
    Step:   959 | Loss: 5.2077 | lr = 0.000060 | Norm = 0.482971 | Time: 35.4782sec | Token/sec: 1847.221
    Step:   960 | Loss: 5.1894 | lr = 0.000060 | Norm = 0.381360 | Time: 35.4482sec | Token/sec: 1848.785
    Step:   961 | Loss: 5.1035 | lr = 0.000060 | Norm = 0.388839 | Time: 35.5325sec | Token/sec: 1844.397
    Step:   962 | Loss: 5.2928 | lr = 0.000060 | Norm = 0.528495 | Time: 35.4737sec | Token/sec: 1847.451
    Step:   963 | Loss: 5.2435 | lr = 0.000060 | Norm = 0.343517 | Time: 35.4399sec | Token/sec: 1849.217
    Step:   964 | Loss: 5.1661 | lr = 0.000060 | Norm = 0.416844 | Time: 35.4682sec | Token/sec: 1847.737
    Step:   965 | Loss: 5.2895 | lr = 0.000060 | Norm = 0.412045 | Time: 35.4437sec | Token/sec: 1849.016
    Step:   966 | Loss: 5.2015 | lr = 0.000060 | Norm = 0.471982 | Time: 35.4481sec | Token/sec: 1848.787
    Step:   967 | Loss: 5.1605 | lr = 0.000060 | Norm = 0.391295 | Time: 35.4850sec | Token/sec: 1846.866
    Step:   968 | Loss: 5.2097 | lr = 0.000060 | Norm = 0.379458 | Time: 35.4486sec | Token/sec: 1848.762
    Step:   969 | Loss: 5.2858 | lr = 0.000060 | Norm = 0.430927 | Time: 35.4536sec | Token/sec: 1848.499
    Step:   970 | Loss: 5.2138 | lr = 0.000060 | Norm = 0.404029 | Time: 35.4880sec | Token/sec: 1846.710
    Step:   971 | Loss: 5.1982 | lr = 0.000060 | Norm = 0.466684 | Time: 35.4573sec | Token/sec: 1848.306
    Step:   972 | Loss: 5.3151 | lr = 0.000060 | Norm = 0.443109 | Time: 35.4622sec | Token/sec: 1848.054
    Step:   973 | Loss: 5.1961 | lr = 0.000060 | Norm = 0.383884 | Time: 35.4602sec | Token/sec: 1848.156
    Step:   974 | Loss: 5.2116 | lr = 0.000060 | Norm = 0.519472 | Time: 35.4826sec | Token/sec: 1846.988
    Step:   975 | Loss: 5.1678 | lr = 0.000060 | Norm = 0.495235 | Time: 35.4879sec | Token/sec: 1846.712
    Step:   976 | Loss: 5.1777 | lr = 0.000060 | Norm = 0.406459 | Time: 35.4572sec | Token/sec: 1848.314
    Step:   977 | Loss: 5.2077 | lr = 0.000060 | Norm = 0.496259 | Time: 35.4835sec | Token/sec: 1846.941
    Step:   978 | Loss: 5.0567 | lr = 0.000060 | Norm = 0.374137 | Time: 35.4710sec | Token/sec: 1847.593
    Step:   979 | Loss: 5.1336 | lr = 0.000060 | Norm = 0.508208 | Time: 35.4691sec | Token/sec: 1847.694
    Step:   980 | Loss: 5.1743 | lr = 0.000060 | Norm = 0.387192 | Time: 35.4048sec | Token/sec: 1851.047
    Step:   981 | Loss: 5.0830 | lr = 0.000060 | Norm = 0.440353 | Time: 35.2511sec | Token/sec: 1859.117
    Step:   982 | Loss: 5.2845 | lr = 0.000060 | Norm = 0.435430 | Time: 35.0457sec | Token/sec: 1870.017
    Step:   983 | Loss: 5.1818 | lr = 0.000060 | Norm = 0.408892 | Time: 35.2915sec | Token/sec: 1856.989
    Step:   984 | Loss: 5.1834 | lr = 0.000060 | Norm = 0.621280 | Time: 35.4642sec | Token/sec: 1847.949
    Step:   985 | Loss: 5.1516 | lr = 0.000060 | Norm = 0.527088 | Time: 35.5476sec | Token/sec: 1843.613
    Step:   986 | Loss: 5.1732 | lr = 0.000060 | Norm = 0.402005 | Time: 35.4453sec | Token/sec: 1848.931
    Step:   987 | Loss: 5.1697 | lr = 0.000060 | Norm = 0.403270 | Time: 35.1698sec | Token/sec: 1863.419
    Step:   988 | Loss: 5.1841 | lr = 0.000060 | Norm = 0.502643 | Time: 35.0922sec | Token/sec: 1867.540
    Step:   989 | Loss: 5.1771 | lr = 0.000060 | Norm = 0.366349 | Time: 35.1623sec | Token/sec: 1863.815
    Step:   990 | Loss: 5.2036 | lr = 0.000060 | Norm = 0.388264 | Time: 35.2779sec | Token/sec: 1857.709
    Step:   991 | Loss: 5.2401 | lr = 0.000060 | Norm = 0.413675 | Time: 35.4266sec | Token/sec: 1849.909
    Step:   992 | Loss: 5.2051 | lr = 0.000060 | Norm = 0.382144 | Time: 35.4368sec | Token/sec: 1849.376
    Step:   993 | Loss: 5.1681 | lr = 0.000060 | Norm = 0.421612 | Time: 35.4912sec | Token/sec: 1846.542
    Step:   994 | Loss: 5.1676 | lr = 0.000060 | Norm = 0.406738 | Time: 35.5063sec | Token/sec: 1845.756
    Step:   995 | Loss: 5.0814 | lr = 0.000060 | Norm = 0.411854 | Time: 35.5429sec | Token/sec: 1843.856
    Step:   996 | Loss: 5.1749 | lr = 0.000060 | Norm = 0.415088 | Time: 35.4683sec | Token/sec: 1847.734
    Step:   997 | Loss: 5.0669 | lr = 0.000060 | Norm = 0.396898 | Time: 35.1940sec | Token/sec: 1862.138
    Step:   998 | Loss: 5.1332 | lr = 0.000060 | Norm = 0.456988 | Time: 35.0284sec | Token/sec: 1870.940
    Validation loss: 5.1265
    HellaSwag accuracy: 2333/10042=0.2323
    Step:   999 | Loss: 5.0966 | lr = 0.000060 | Norm = 0.398870 | Time: 365.3984sec | Token/sec: 179.355

Here not only the master process makes the sampling also other processes, it is not considered a bug, but like can be made better (e.g. only sample at the master process)